Silver Summary Table QA

# Daily KPI Summary Validation Notebook

This notebook validates the daily KPI summary table used in the Topline dashboard.  
It compares key metrics from the `SILVER_EVENT_DIM` table with those in the summary table to ensure consistency and accuracy.

## Items Checked

### Daily KPI

- **A1. Overall Metrics**: PV/U, PV/S, Session/User, Article Bottom Reach %, Article Recirculation %, Video Completion %
- **A2. Metrics by Day**: PV/U, PV/S, Session/User, Article Bottom Reach %, Article Recirculation %, Video Completion %
- **A3. Metrics by Source**: PV/U, PV/S, Session/User, Article Bottom Reach %, Article Recirculation %, Video Completion %
- **A4. Metrics by Country**: PV/U, PV/S, Session/User, Article Bottom Reach %, Article Recirculation %, Video Completion %
- **A5. Metrics by User Type**: PV/U, PV/S, Session/User, Article Bottom Reach %, Article Recirculation %, Video Completion %
- **A6. Metrics by Edition**: PV/U, PV/S, Session/User, Article Bottom Reach %, Article Recirculation %, Video Completion %
- **A7. Metrics by Simplified Referrer**: PV/U, PV/S, Session/User, Article Bottom Reach %, Article Recirculation %, Video Completion %
- **A8. Metrics by Region**: PV/U, PV/S, Session/User, Article Bottom Reach %, Article Recirculation %, Video Completion %

- **B1. Metrics by Source and Referrer**

- **C1. Metrics by Day and Source**  
- **C2. Metrics by Day and User Type**  
- **C3. Metrics by User Type and Source**  
- **C4. Metrics by User Type and Referrer**  
- **C5. Metrics by User Type and Country**

- **D1. Metrics by Day, Source, Referrer, Edition**  
- **D2. Metrics by Day, Source, User Type, Referrer, Country**

- **E1. Daily Trend for Date Range Jan 4–6**  
- **E2. Sessions by Referrer Pie Chart**

### Content

- **A1. Overall**  
- **A2. By Day**  
- **A3. By Topic Channel**  
- **A4. By User Type**  
- **A5. By Country**  
- **A6. By Region**  
- **A7. By Subschannel**  
- **A8. By Referrer**  
- **A9. By Edition**

- **B1. Users and PV of Top 10 Canonical URLs (Jan 5–9)**  
- **B2. Users and PV of Top 10 Topic Channels (Jan 5–9)**  
- **B3. Users and PV of Top 10 Landing Pages (Jan 5–9)**  
- **B4. Users and PV of Top 10 Articles (Jan 5–9)**


In [1]:
# Import libraries 
import pandas as pd
from dotenv import load_dotenv
import numpy as np
import snowflake.connector
import sys

In [ ]:
# Set up the data connection 
con = snowflake.connector.connect(
    user="joonkyung.kim@thomsonreuters.com", #You can get it by executing in UI: desc user <username>;
    account="THOMSONREUTERS-A206448_PROD", #Add all of the account-name between https:// and snowflakecomputing.com
    authenticator="externalbrowser",
)
cur = con.cursor()

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://sso.thomsonreuters.com/idp/SSO.saml2?SAMLRequest=nZJNc9owEIb%2Fikc9W%2F7AUKIBMjSElJk0UHDaTC8d1VpAgy25WimG%2FvoKE8%2Bkh%2BTQm0d%2BdvWs3h1dH6syeAaDUqsxSWhMAlCFFlLtxuQxn4dDEqDlSvBSKxiTEyC5noyQV2XNps7u1Rp%2BO0Ab%2BEYKWftjTJxRTHOUyBSvAJkt2Gb65Z6lNGa10VYXuiSvSt6v4IhgrDfsSgRKr7e3tmZR1DQNbXpUm12UxnEcxVeRp87Ih44%2F%2Bpne4JMozs68Jzy%2BenH7JNXlCd7T%2BnWBkH3O81W4Wm5yEkw71Rut0FVgNmCeZQGP6%2FuLAHqDaRoPsmz4c7VezqjDEDjaMKGodLMt%2BQEKXdXO%2BtbUf0VbEFGpd9JPv5iNSX2QQhyzp5NQ8DT4M2%2Bwr%2B%2BSLHffB25Q2Nv1He%2F3ZL8v4OF2ffhakOBbF296jneB6GChzqFafxSngzDOwuRjngxZ0mPpFU2z5AcJZj5UqbhtKztzRE3tXleolQFnfeNWUYo62myW9BxlSi7rwdqLzOS%2Fhh5Fr1u8rNuDT2AxW%2BlSFqdgrk3F7dsBJTRpT6QIty3KoOKynAphANEHVZa6uTHArd9qaxyQaHK59d%2B9nvwF&RelayState=ver%3A3-hint%3A7475593065661118-ETMsDgAAAZ2cpd4ZABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEHl1QiAfBFxeZn5gRjf1MjcAAACw4tp%2BKLK17UoBMwUy7cEGXWOaJ6%2BsFHJtKephve9nt3gQdLO0okgMVEWWU3Z

In [3]:
start_date = '2026-03-15'
end_date = '2026-03-20'
source_filter = "AND E.SOURCE IN ('desktop', 'mobileweb')"
daily_kpi_table_name = 'SILVER_DAILY_KPI_SUMMARY'
daily_content_table_name = 'SILVER_DAILY_CONTENT_SUMMARY'
date_set_a_start = '2026-01-01'
date_set_a_end = '2026-01-02'
date_set_b_start = '2026-01-03'
date_set_b_end = '2026-01-04'
date_range = "('2025-01-01', '2025-01-02', '2025-01-03', '2025-01-04', '2025-01-05', '2025-01-06', '2025-01-07', '2025-01-08', '2025-01-09', '2025-01-10')" #,'2025-02-02', '2025-03-01')"
ACCEPTABLE_THRESHOLD = 0.1  # percent (1 / 1000)
METRIC_COLS = [
    "USER_COUNT",
    "SESSIONS",
    "PV",
    "VIDEO_STORY_START",
    "VIDEO_STORY_FINISHED",
    "ARTICLE_RECIRCULATION",
    "ARTICLE_BOTTOM_REACH"
]

In [4]:
# cur.execute("""USE ROLE A208946_REUTERS_ANALYTICS_MDS_READ_WRITE""")
cur.execute("""USE ROLE REUTERS_CDP_QA_ANALYTICS""")
cur.execute("""USE WAREHOUSE A208946_REUTERS_ANALYTICS_MDS_WH""")

## Functions

In [5]:
def compare_dataframes(df1, df2, column_list, content_flag=False):

    # ---------------------------------------------------
    # Statement 1: Row count comparison
    # ---------------------------------------------------
    rows_df1 = len(df1)
    rows_df2 = len(df2)

    row_match_status = "match" if rows_df1 == rows_df2 else "do not match"
    print(
        f"The number of rows in df1 and df2 are {rows_df1} and {rows_df2}. "
        f"They {row_match_status}."
    )

    metrics = list(set(METRIC_COLS) & set(df1.columns))

    if content_flag:    
        sorted_df1 = df1.sort_values(by=column_list).reset_index(drop=True)
        sorted_df2 = df2.sort_values(by=column_list).reset_index(drop=True)
        comparison_result = sorted_df1.fillna(np.nan).equals(sorted_df2.fillna(np.nan))
        if comparison_result:
            return 'PASS'
        else:
            return 'FAIL'
        

    # ---------------------------------------------------
    # Step 1: Sort + null-safe normalization
    # ---------------------------------------------------
    sorted_df1 = df1.sort_values(by=column_list).reset_index(drop=True).fillna(np.nan)
    sorted_df2 = df2.sort_values(by=column_list).reset_index(drop=True).fillna(np.nan)

    # ---------------------------------------------------
    # Step 2: Aggregate metrics
    # ---------------------------------------------------
    df1_grp = (
        sorted_df1
        .groupby(column_list, dropna=False)[metrics]
        .sum()
        .reset_index()
    )

    df2_grp = (
        sorted_df2
        .groupby(column_list, dropna=False)[metrics]
        .sum()
        .reset_index()
    )

    # ---------------------------------------------------
    # Step 3: Combination parity check
    # ---------------------------------------------------
    if not df1_grp[column_list].equals(df2_grp[column_list]):
        print("❌ Value combinations do not match between df1 and df2")
        return "FAIL"

    # ---------------------------------------------------
    # Step 4: Merge for comparison
    # ---------------------------------------------------
    comparison_df = df1_grp.merge(
        df2_grp,
        on=column_list,
        how="inner",
        suffixes=("_df1", "_df2")
    )

    diff_rows = []
    metric_summary = {}

    # ---------------------------------------------------
    # Step 5: Metric-level comparison + % difference
    # ---------------------------------------------------
    for metric in metrics:
        col1 = f"{metric}_df1"
        col2 = f"{metric}_df2"

        comparison_df["difference"] = comparison_df[col1] - comparison_df[col2]
        comparison_df["%_DIFFERENCE"] = np.where(
            comparison_df[col1] == 0,
            np.nan,
            (comparison_df["difference"] / comparison_df[col1]).abs() * 100
        )

        metric_diffs = comparison_df[comparison_df["difference"] != 0]

        if not metric_diffs.empty:
            min_diff = metric_diffs["%_DIFFERENCE"].min()
            max_diff = metric_diffs["%_DIFFERENCE"].max()

            metric_summary[metric] = (min_diff, max_diff)

            diff_output = metric_diffs.copy()
            diff_output["metric"] = metric
            diff_output["df1_value"] = diff_output[col1]
            diff_output["df2_value"] = diff_output[col2]

            diff_rows.append(
                diff_output[
                    column_list
                    + ["metric", "df1_value", "df2_value", "difference", "%_DIFFERENCE"]
                ]
            )

    # ---------------------------------------------------
    # Statement 2: Metric discrepancy summary
    # ---------------------------------------------------
    for metric, (min_diff, max_diff) in metric_summary.items():
        acceptability = (
            "This is acceptable."
            if max_diff < ACCEPTABLE_THRESHOLD
            else "This is not acceptable."
        )

        print(
            f"The discrepancy in metric {metric} ranges between "
            f"{min_diff:.4f}% and {max_diff:.4f}%. "
            f"The absolute discrepancy is {'below' if max_diff < ACCEPTABLE_THRESHOLD else 'above'} "
            f"{ACCEPTABLE_THRESHOLD}%. {acceptability}"
        )

    # ---------------------------------------------------
    # Final output
    # ---------------------------------------------------
    if diff_rows:
        detailed_diff = pd.concat(diff_rows, ignore_index=True)
        print("\nDetailed discrepancies:\n", detailed_diff)

    for metric, (min_diff, max_diff) in metric_summary.items():
        if max_diff < ACCEPTABLE_THRESHOLD:
            status = "PASS"
            message = "✅ This is acceptable."
        else:
            status = "FAIL"
            message = "❌ This is not acceptable."

        print(
            f"The discrepancy in metric {metric} ranges between "
            f"{min_diff:.4f}% and {max_diff:.4f}%. "
            f"The absolute discrepancy is "
            f"{'below' if status == 'PASS' else 'above'} "
            f"{ACCEPTABLE_THRESHOLD}%. {message} [{status}]"
        )


def compare_dataframes_bk(df1, df2, column_list, content_flag):
    # Step 1: Sort both dataframes
    sorted_df1 = df1.sort_values(by=column_list).reset_index(drop=True)
    sorted_df2 = df2.sort_values(by=column_list).reset_index(drop=True)

    if content_flag:
        comparison_result = sorted_df1.fillna(np.nan).equals(sorted_df2.fillna(np.nan))
        if comparison_result:
            return 'PASS'
        else:
            return 'FAIL'
    
    else:
        # Step 2: Handle null-safe comparison
        comparison_result = sorted_df1.fillna(np.nan).equals(sorted_df2.fillna(np.nan))

        # Step 3: Group by column_list and sum 'SESSIONS'
        df1_grouped_sessions = df1.groupby(column_list)['SESSIONS'].sum()
        df2_grouped_sessions = df2.groupby(column_list)['SESSIONS'].sum()

        # Step 4: Create comparison DataFrame
        comparison = pd.DataFrame({
            'SESSIONS_df1': df1_grouped_sessions,
            'SESSIONS_df2': df2_grouped_sessions
        })

        # Step 5: Calculate percentage difference
        comparison['%_DIFFERENCE'] = ((comparison['SESSIONS_df1'] - comparison['SESSIONS_df2']) / comparison['SESSIONS_df1']) * 100
        print("\nSession comparison:\n", comparison)

        # Step 6: Drop 'SESSIONS' column for final comparison
        df1_excluded = sorted_df1.drop(columns=['SESSIONS'])
        df2_excluded = sorted_df2.drop(columns=['SESSIONS'])

        # Step 7: Null-safe comparison for excluded DataFrames
        null_safe_equal = df1_excluded.fillna(np.nan).equals(df2_excluded.fillna(np.nan))

        # Step 8: Final pass/fail check
        if (comparison['%_DIFFERENCE'].abs() < 1).all() and null_safe_equal:
            print('PASS - difference in sessions is less than 1% and dataframes match')
        else:
            print('FAIL - either session difference exceeds 1% or dataframes do not match')

def normalize(val):
    return '' if pd.isna(val) or val == '' else str(val)

def compare_monthly_dataframes(df1, df2, column_list, threshold):
    null_safe_equal = False
    # Step 1: Sort both dataframes
    placeholder = ''
    df1_safe = df1.fillna(placeholder)
    df2_safe = df2.fillna(placeholder)

    sorted_df1 = df1_safe.sort_values(by=column_list).reset_index(drop=True)
    sorted_df2 = df2_safe.sort_values(by=column_list).reset_index(drop=True)

    # Step 6: Drop 'SESSIONS' column for final comparison
    df1_excluded = sorted_df1.drop(columns=['SESSIONS'])
    df2_excluded = sorted_df2.drop(columns=['SESSIONS'])

    # Step 7: Null-safe comparison for excluded DataFrames
    null_safe_equal_excl_sessions = (df1_excluded.astype(str) == df2_excluded.astype(str)).all().all()

    # Step 3: Group by column_list and sum 'SESSIONS'
    df1_grouped_sessions = sorted_df1.groupby(column_list)['SESSIONS'].sum()
    df2_grouped_sessions = sorted_df2.groupby(column_list)['SESSIONS'].sum()

    # Step 4: Create comparison DataFrame
    comparison = pd.DataFrame({
        'SESSIONS_df1': df1_grouped_sessions,
        'SESSIONS_df2': df2_grouped_sessions
    })

    # Step 5: Calculate percentage difference
    comparison['SESSIONS_df1'] = comparison['SESSIONS_df1'].astype(float)
    comparison['SESSIONS_df2'] = comparison['SESSIONS_df2'].astype(float)

        
    comparison['DIFF'] = (comparison['SESSIONS_df1'] - comparison['SESSIONS_df2']).abs()
    comparison['%_DIFFERENCE'] = ((comparison['SESSIONS_df1'] - comparison['SESSIONS_df2']) / comparison['SESSIONS_df1']) * 100
    print("\nSession comparison:\n", comparison)

    if ((comparison['DIFF'] < threshold).all() or (comparison['%_DIFFERENCE'].abs() < 1).all()) and null_safe_equal_excl_sessions:
        null_safe_equal = True

    #null_safe_equal = (sorted_df1.astype(str) == sorted_df2.astype(str)).all().all()

    # Step 8: Final pass/fail check
    if null_safe_equal:
        return 'PASS'
    else:
        return 'FAIL'


def compare_monthly_content(df1, df2, column_list, threshold):
    null_safe_equal = False
    # Step 1: Sort both dataframes
    #placeholder = ''
    #df1_safe = df1.fillna(placeholder)
    #df2_safe = df2.fillna(placeholder)
    
    df1_safe = df1.apply(lambda col: col.map(normalize))
    df2_safe = df2.apply(lambda col: col.map(normalize))

    sorted_df1 = df1_safe.sort_values(by=column_list).reset_index(drop=True)
    sorted_df2 = df2_safe.sort_values(by=column_list).reset_index(drop=True)
    null_safe_equal = (sorted_df1.astype(str) == sorted_df2.astype(str)).all().all()
    
    """
    if 'USER_TYPE' in column_list:
        # Step 6: Drop 'SESSIONS' column for final comparison
        df1_excluded = sorted_df1.drop(columns=['SESSIONS'])
        df2_excluded = sorted_df2.drop(columns=['SESSIONS'])

        # Step 7: Null-safe comparison for excluded DataFrames
        null_safe_equal_excl_sessions = (df1_excluded.astype(str) == df2_excluded.astype(str)).all().all()

        # Step 3: Group by column_list and sum 'SESSIONS'
        df1_grouped_sessions = sorted_df1.groupby(column_list)['SESSIONS'].sum()
        df2_grouped_sessions = sorted_df2.groupby(column_list)['SESSIONS'].sum()

        # Step 4: Create comparison DataFrame
        comparison = pd.DataFrame({
            'SESSIONS_df1': df1_grouped_sessions,
            'SESSIONS_df2': df2_grouped_sessions
        })

        # Step 5: Calculate percentage difference
        comparison['SESSIONS_df1'] = comparison['SESSIONS_df1'].astype(float)
        comparison['SESSIONS_df2'] = comparison['SESSIONS_df2'].astype(float)

        
        comparison['DIFF'] = (comparison['SESSIONS_df1'] - comparison['SESSIONS_df2']).abs()
        comparison['%_DIFFERENCE'] = ((comparison['SESSIONS_df1'] - comparison['SESSIONS_df2']) / comparison['SESSIONS_df1']) * 100
        print("\nSession comparison:\n", comparison)

        if ((comparison['DIFF'] < threshold).all() or (comparison['%_DIFFERENCE'].abs() < 1).all()) and null_safe_equal_excl_sessions:
            null_safe_equal = True

    else:
        null_safe_equal = (sorted_df1.astype(str) == sorted_df2.astype(str)).all().all()
    """
    # Step 8: Final pass/fail check
    if null_safe_equal:
        return 'PASS'
    else:
        return 'FAIL'
    
def run_query(query):
    cur.execute(query)
    rows = cur.fetchall()
    cols = [col[0] for col in cur.description]
    df = pd.DataFrame(rows, columns=cols)
    #df = cur.fetch_pandas_all()

    metric_cols = [
        'USER_COUNT', 'SESSIONS', 'PAGEVIEWS',
        'VIDEO_STORY_START', 'VIDEO_STORY_FINISHED',
        'ARTICLE_RECIRCULATION', 'ARTICLE_BOTTOM_REACH'
    ]

    # Only process columns that exist in the dataframe
    for col in (c for c in metric_cols if c in df.columns):
        if df[col].dtype == 'object':
            df[col] = df[col].astype('int64')
    return df
    
def derived_metrics(df1):
    df1['PV/U'] = df1['PV'] / df1['USER_COUNT']
    df1['PV/S'] = df1['PV'] / df1['SESSIONS']
    df1['sessions/U'] = df1['SESSIONS'] / df1['USER_COUNT']
    df1['bottom_reach_perc'] = df1['ARTICLE_BOTTOM_REACH'] / df1['PV']
    df1['recirculation_perc'] = df1['ARTICLE_RECIRCULATION'] / df1['PV']
    df1['video_completion_perc'] = df1['VIDEO_STORY_FINISHED'] / df1['VIDEO_STORY_START']
    
    return df1


## Query for Daily KPI

In [336]:
q1_overall = f"""select
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count, --count(distinct e.ss_anonymous_user_id) as user_count,
    count(distinct ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', ss_event_id, null)) as pv,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
where d between '{start_date}' and '{end_date}' and segment_user_agent_type='other user agent' {source_filter}"""

q1s_overall = f"""select
    count(distinct ADJ_COALESCE_UID) AS USER_COUNT, --COALESCE(UID_BY_USERTYPE, coalesce_uid)) as user_count,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv,
    sum(video_story_start) AS VIDEO_STORY_START,
    sum(video_story_finished) AS VIDEO_STORY_FINISHED,
    sum(article_recirculation) AS ARTICLE_RECIRCULATION,
    sum(article_bottom_reach) AS ARTICLE_BOTTOM_REACH 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_kpi_table_name}
where d between '{start_date}' and '{end_date}' """

q2_daily = f"""select d,
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count, --count(distinct e.ss_anonymous_user_id) as user_count,
    count(distinct ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', ss_event_id, null)) as pv,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
where d between '{start_date}' and '{end_date}' and segment_user_agent_type='other user agent' {source_filter} group by 1"""

q2s_daily = f"""select d,
    count(distinct ADJ_COALESCE_UID) AS USER_COUNT,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv,
    sum(video_story_start) AS VIDEO_STORY_START,
    sum(video_story_finished) AS VIDEO_STORY_FINISHED,
    sum(article_recirculation) AS ARTICLE_RECIRCULATION,
    sum(article_bottom_reach) AS ARTICLE_BOTTOM_REACH
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_kpi_table_name}
where d between '{start_date}' and '{end_date}' group by 1"""

q3_by_source = f"""select source, 
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count, --count(distinct e.ss_anonymous_user_id) as user_count,
    count(distinct ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', ss_event_id, null)) as pv,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e 
where d between '{start_date}' and '{end_date}' and segment_user_agent_type='other user agent' {source_filter} group by 1"""

q3s_by_source = f"""select source,
    count(distinct ADJ_COALESCE_UID) AS USER_COUNT,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv,
    sum(video_story_start) AS VIDEO_STORY_START,
    sum(video_story_finished) AS VIDEO_STORY_FINISHED,
    sum(article_recirculation) AS ARTICLE_RECIRCULATION,
    sum(article_bottom_reach) AS ARTICLE_BOTTOM_REACH
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_kpi_table_name}
where d between '{start_date}' and '{end_date}' group by 1"""

q4_by_country = f"""select c.country_name as FIRST_COUNTRY,
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count,
    count(distinct E.ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', ss_event_id, null)) as pv,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.ss_SESSION_ID = S.ss_SESSION_ID
    LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' 
    and e.segment_user_agent_type='other user agent' {source_filter} group by 1"""

q4s_by_country = f"""select FIRST_COUNTRY,
    count(distinct ADJ_COALESCE_UID) AS USER_COUNT,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv,
    sum(video_story_start) AS VIDEO_STORY_START,
    sum(video_story_finished) AS VIDEO_STORY_FINISHED,
    sum(article_recirculation) AS ARTICLE_RECIRCULATION,
    sum(article_bottom_reach) AS ARTICLE_BOTTOM_REACH 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_kpi_table_name}
where d between '{start_date}' and '{end_date}' group by 1"""

q5_by_user_type = f"""select
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
            ELSE NULL END AS USER_TYPE,
    count(distinct IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register'), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct E.SS_session_id) as sessions,
    count(distinct iff(e.event_type = 'page', ss_event_id, null)) as pv,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
where e.d between '{start_date}' and '{end_date}' 
    and e.segment_user_agent_type='other user agent' {source_filter} group by 1"""

q5s_by_user_type = f"""select USER_TYPE,
    count(distinct UID_BY_USERTYPE) as user_count,
    sum(SESSIONS_BY_USERTYPE) as sessions,
    sum(pageviews) as pv,
    sum(video_story_start) AS VIDEO_STORY_START,
    sum(video_story_finished) AS VIDEO_STORY_FINISHED,
    sum(article_recirculation) AS ARTICLE_RECIRCULATION,
    sum(article_bottom_reach) AS ARTICLE_BOTTOM_REACH 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_kpi_table_name}
where d between '{start_date}' and '{end_date}' group by 1"""

q6_by_edition = f"""select
    ED.EDITION as first_edition,
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count,
    count(distinct E.SS_session_id) as sessions,
    count(distinct iff(event_type = 'page', ss_event_id, null)) as pv,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.SS_EDITION_ID = ED.SS_EDITION_ID
where e.d between '{start_date}' and '{end_date}' 
    and e.segment_user_agent_type='other user agent' {source_filter} group by 1"""

q6s_by_edition = f"""select first_edition,
    count(distinct ADJ_COALESCE_UID) AS USER_COUNT,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv,
    sum(video_story_start) AS VIDEO_STORY_START,
    sum(video_story_finished) AS VIDEO_STORY_FINISHED,
    sum(article_recirculation) AS ARTICLE_RECIRCULATION,
    sum(article_bottom_reach) AS ARTICLE_BOTTOM_REACH 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_kpi_table_name}
where d between '{start_date}' and '{end_date}' group by 1"""

q7_by_referrer = f"""select
    COALESCE(TRIM(REGEXP_SUBSTR(FIRST_UTM_REFERRER_NULL_ALLOWED, '^[^-]+')), 'Other') AS FIRST_UTM_REFERRER_SIMPLIFIED, 
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count,
    count(distinct E.ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', ss_event_id, null)) as pv,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
where e.d between '{start_date}' and '{end_date}' 
    and e.segment_user_agent_type='other user agent' {source_filter} group by 1"""

q7s_by_referrer = f"""select
    FIRST_UTM_REFERRER_SIMPLIFIED, 
    count(distinct ADJ_COALESCE_UID) AS USER_COUNT,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv,
    sum(video_story_start) AS VIDEO_STORY_START,
    sum(video_story_finished) AS VIDEO_STORY_FINISHED,
    sum(article_recirculation) AS ARTICLE_RECIRCULATION,
    sum(article_bottom_reach) AS ARTICLE_BOTTOM_REACH 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_kpi_table_name}
where d between '{start_date}' and '{end_date}'  group by 1"""

q8_by_region = f"""select
    c.region as first_region,
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count,
    count(distinct E.ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', ss_event_id, null)) as pv,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_SS_COUNTRY_ID = C.SS_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}'  
    and e.segment_user_agent_type='other user agent' {source_filter} group by 1"""

q8s_by_region = f"""select
    FIRST_REGION,
    count(distinct ADJ_COALESCE_UID) AS USER_COUNT,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv,
    sum(video_story_start) AS VIDEO_STORY_START,
    sum(video_story_finished) AS VIDEO_STORY_FINISHED,
    sum(article_recirculation) AS ARTICLE_RECIRCULATION,
    sum(article_bottom_reach) AS ARTICLE_BOTTOM_REACH 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_kpi_table_name}
where d between '{start_date}' and '{end_date}'  group by 1"""

q9_by_continent = f"""select
    c.region_continent as first_continent,
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count,
    count(distinct E.ss_session_id) as sessions,
    count(distinct iff(event_type = 'page', ss_event_id, null)) as pv,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_SS_COUNTRY_ID = C.SS_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}'  
    and e.segment_user_agent_type='other user agent' {source_filter} group by 1"""

q9s_by_continent = f"""select
    FIRST_continent,
    count(distinct ADJ_COALESCE_UID) AS USER_COUNT,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv,
    sum(video_story_start) AS VIDEO_STORY_START,
    sum(video_story_finished) AS VIDEO_STORY_FINISHED,
    sum(article_recirculation) AS ARTICLE_RECIRCULATION,
    sum(article_bottom_reach) AS ARTICLE_BOTTOM_REACH 
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_kpi_table_name}
where d between '{start_date}' and '{end_date}'  group by 1"""

In [337]:
b1_by_source_referrer = f"""select
    S.source,
    COALESCE(TRIM(REGEXP_SUBSTR(FIRST_UTM_REFERRER_NULL_ALLOWED, '^[^-]+')), 'Other') AS FIRST_UTM_REFERRER_SIMPLIFIED, 
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count,
    count(distinct E.SS_session_id) as sessions,
    count(distinct iff(event_type = 'page', ss_event_id, null)) as pv
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
where e.d between '{start_date}' and '{end_date}' 
    and e.segment_user_agent_type='other user agent' {source_filter} group by 1,2"""

b1s_by_source_referrer = f"""select
    source,
    FIRST_UTM_REFERRER_SIMPLIFIED, 
    count(distinct ADJ_COALESCE_UID) AS USER_COUNT,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_kpi_table_name}
where d between '{start_date}' and '{end_date}' group by 1,2"""

c1_by_day_source = f"""select
    e.d, s.source,
    count(distinct coalesce(e.user_dim_id, e.ss_anonymous_user_id)) as user_count,
    count(distinct E.ss_session_id) as sessions,
    count(distinct iff(E.event_type = 'page', ss_event_id, null)) as pv
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
where e.d between '{start_date}' and '{end_date}' 
    and e.segment_user_agent_type='other user agent' {source_filter} group by 1,2"""

c1s_by_day_source = f"""select d,
    source,
    count(distinct ADJ_COALESCE_UID) AS USER_COUNT,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_kpi_table_name}
where d between '{start_date}' and '{end_date}' group by 1,2"""

# BY USER TYPE AND DAY
c2_by_day_usertype = f"""select e.d,
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
            ELSE NULL END AS USER_TYPE,
    count(distinct IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register'), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct E.SS_SESSION_ID) as sessions,
    count(distinct iff(E.event_type = 'page', ss_event_id, null)) as pv
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.ss_EDITION_ID = ED.ss_EDITION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_ss_COUNTRY_ID = C.ss_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' and e.segment_user_agent_type='other user agent' {source_filter} group by 1,2"""

c2s_by_day_usertype = f"""select d,
    USER_TYPE,
    count(distinct uid_by_usertype) as user_count,
    sum(SESSIONS_BY_USERTYPE) as sessions,
    sum(pageviews) as pv
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_kpi_table_name}
where d between '{start_date}' and '{end_date}' group by 1,2"""

# BY USER TYPE AND SOURCE
c3_by_source_usertype = f"""select 
    s.source,
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
            ELSE NULL END AS USER_TYPE,
    count(distinct IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register'), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct E.SS_SESSION_ID) as sessions,
    count(distinct iff(E.event_type = 'page', ss_event_id, null)) as pv
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.SS_EDITION_ID = ED.SS_EDITION_ID
where e.d between '{start_date}' and '{end_date}' and e.segment_user_agent_type='other user agent' {source_filter} group by 1,2"""

c3s_by_source_usertype = f"""select 
    source,
    USER_TYPE,
    count(distinct uid_by_usertype) as user_count,
    sum(SESSIONS_BY_USERTYPE) as sessions,
    sum(pageviews) as pv
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_kpi_table_name}
where d between '{start_date}' and '{end_date}' group by 1,2"""

# BY USER TYPE AND REFERRER
c4_by_usertype_referrer = f"""select 
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
            ELSE NULL END AS USER_TYPE,
    COALESCE(TRIM(REGEXP_SUBSTR(FIRST_UTM_REFERRER_NULL_ALLOWED, '^[^-]+')), 'Other') AS FIRST_UTM_REFERRER_SIMPLIFIED, 
    count(distinct IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register'), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct E.SS_SESSION_ID) as sessions,
    count(distinct iff(E.event_type = 'page', ss_event_id, null)) as pv
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON E.SS_EDITION_ID = ED.SS_EDITION_ID
where e.d between '{start_date}' and '{end_date}' and e.segment_user_agent_type='other user agent' {source_filter} group by 1,2"""

c4s_by_usertype_referrer = f"""select 
    USER_TYPE,
    FIRST_UTM_REFERRER_SIMPLIFIED, 
    count(distinct uid_by_usertype) as user_count,
    sum(SESSIONS_BY_USERTYPE) as sessions,
    sum(pageviews) as pv
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_kpi_table_name}
where d between '{start_date}' and '{end_date}' group by 1,2"""

# BY USER TYPE AND COUNTRY
c5_by_usertype_country = f"""select 
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
            WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
            ELSE NULL END AS USER_TYPE,
    C.COUNTRY_NAME AS FIRST_COUNTRY,
    count(distinct IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register'), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID)) as user_count,
    count(distinct E.SS_SESSION_ID) as sessions,
    count(distinct iff(E.event_type = 'page', ss_event_id, null)) as pv
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.SS_EDITION_ID = ED.SS_EDITION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_SS_COUNTRY_ID = C.SS_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' and e.segment_user_agent_type='other user agent' {source_filter} group by 1,2"""

c5s_by_usertype_country = f"""select 
    USER_TYPE,
    FIRST_COUNTRY,
    count(distinct uid_by_usertype) as user_count,
    sum(SESSIONS_BY_USERTYPE) as sessions,
    sum(pageviews) as pv
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_kpi_table_name}
where d between '{start_date}' and '{end_date}' group by 1,2"""

d1_by_day_source_referrer_edition = f"""select e.d,
    s.source,
    COALESCE(TRIM(REGEXP_SUBSTR(FIRST_UTM_REFERRER_NULL_ALLOWED, '^[^-]+')), 'Other') AS FIRST_UTM_REFERRER_SIMPLIFIED, 
    ED.EDITION AS FIRST_EDITION,
    count(distinct COALESCE(E.USER_DIM_ID, E.ss_anonymous_user_id)) as user_count,
    count(distinct E.SS_SESSION_ID) as sessions,
    count(distinct iff(E.event_type = 'page', ss_event_id, null)) as pv
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.SS_EDITION_ID = ED.SS_EDITION_ID
where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent' {source_filter} group by 1,2,3,4"""

d1s_by_day_source_referrer_edition = f"""select d,
    source,
    FIRST_UTM_REFERRER_SIMPLIFIED, 
    FIRST_EDITION,
    count(distinct ADJ_COALESCE_UID) AS USER_COUNT,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_kpi_table_name}
where d between '{start_date}' and '{end_date}' group by 1,2,3,4"""

# BY D, USER TYPE, SOURCE, REFERRER
d2_by_day_source_referrer_region = f"""select e.d,
    s.source,
    COALESCE(TRIM(REGEXP_SUBSTR(FIRST_UTM_REFERRER_NULL_ALLOWED, '^[^-]+')), 'Other') AS FIRST_UTM_REFERRER_SIMPLIFIED, 
    C.REGION AS FIRST_REGION,
    count(distinct COALESCE(E.USER_DIM_ID, E.ss_anonymous_user_id)) as user_count,
    count(distinct E.SS_SESSION_ID) as sessions,
    count(distinct iff(E.event_type = 'page', ss_event_id, null)) as pv
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_SS_COUNTRY_ID = C.SS_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent' {source_filter} group by 1,2,3,4"""

d2s_by_day_source_referrer_region = f"""select d,
    source,
    FIRST_UTM_REFERRER_SIMPLIFIED, 
    FIRST_REGION,
    count(distinct ADJ_COALESCE_UID) AS USER_COUNT,
    sum(SESSIONS_for_usertype_all) as sessions,
    sum(pageviews) as pv
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_kpi_table_name}
where d between '{start_date}' and '{end_date}' group by 1,2,3,4"""

# To review the % change
e1 = f"""select 
    iff(e.d between '{date_set_a_start}' and '{date_set_a_end}', 'set_a', iff(e.d between '{date_set_b_start}' and '{date_set_b_end}', 'set_b', null)) date_set,
    s.source,
    C.region AS FIRST_REGION,
    count(distinct E.SS_ANONYMOUS_USER_ID) as user_count,
    count(distinct E.SS_SESSION_ID) as sessions,
    count(distinct iff(E.event_type = 'page', ss_event_id, null)) as pv,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.started', E.ss_event_ID, NULL)) AS VIDEO_STORY_START,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'video_player.story.finished', E.ss_event_ID, NULL)) AS VIDEO_STORY_FINISHED,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.recirculation', E.ss_event_ID, NULL)) AS ARTICLE_RECIRCULATION,
    COUNT(DISTINCT IFF(E.EVENT_NAME = 'article.content.bottom.visible', E.ss_event_ID, NULL)) AS ARTICLE_BOTTOM_REACH 
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_SS_COUNTRY_ID = C.SS_COUNTRY_ID
where e.d between '{date_set_a_start}' and '{date_set_b_end}' {source_filter} and e.segment_user_agent_type='other user agent' group by 1,2,3"""

e1s = f"""select
    iff(d between '{date_set_a_start}' and '{date_set_a_end}', 'set_a', iff(d between '{date_set_b_start}' and '{date_set_b_end}', 'set_b', null)) date_set,
    source,
    FIRST_REGION,
    count(distinct ADJ_COALESCE_UID) AS USER_COUNT,
    sum(SESSIONS_FOR_USERTYPE_ALL) as sessions,
    sum(pageviews) as pv,
    sum(video_story_start) AS VIDEO_STORY_START,
    sum(video_story_finished) AS VIDEO_STORY_FINISHED,
    sum(article_recirculation) AS ARTICLE_RECIRCULATION,
    sum(article_bottom_reach) AS ARTICLE_BOTTOM_REACH
FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_kpi_table_name}
where d between '{date_set_a_start}' and '{date_set_b_end}' group by 1,2,3"""

## Query for daily content 

In [338]:
# VERSION 2 - Dec 09
# by topic channel, by user type, overall, by day, ETC. 
c_a1_overall = f"""select
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', SS_event_id, null)) as pv,
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
where d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent' {source_filter};"""

c_a1s_overall = f"""
    with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_content_table_name} where pageviews > 0)
    select 
    count(distinct COALESCE_UID) as user_count,
    sum(pageviews) as pv
FROM t1
where d between '{start_date}' and '{end_date}';"""

c_a2_by_day = f"""select d,
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', SS_event_id, null)) as pv,
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
where d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent' {source_filter}
    group by 1;"""

c_a2s_by_day = f"""
    with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_content_table_name} where pageviews > 0)
    select d,
    count(distinct COALESCE_UID) as user_count,
    sum(pageviews) as pv
FROM t1
where d between '{start_date}' and '{end_date}' group by 1;"""

c_a3_by_topic_ch = f"""select topic_channel,
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', SS_event_id, null)) as pv,
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
where d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent'  {source_filter}
    group by 1 having pv > 0;"""

c_a3s_by_topic_ch = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_content_table_name} where pageviews > 0)
    select topic_channel,
    count(distinct COALESCE_UID) as user_count,
    sum(pageviews) as pv
FROM t1
where d between '{start_date}' and '{end_date}' group by 1;"""

c_a4_by_usertype = f"""select 
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
        WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
        WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
        ELSE NULL END AS user_type,
    count(distinct iff(event_type = 'page', IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register'), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID), null)) as user_count,
    count(distinct iff(event_type = 'page', SS_event_id, null)) as pv,
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
where d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent' {source_filter}
    group by 1 HAVING PV > 0;"""

c_a4s_by_usertype = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_content_table_name} where pageviews > 0) select user_type,
    count(distinct uid_by_usertype) as user_count,
    sum(pageviews) as pv
FROM t1
where d between '{start_date}' and '{end_date}' group by 1;"""

c_a5_by_country_tch = f"""select 
    c.country_name as first_country,
    topic_channel,
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', e.SS_event_id, null)) as pv,
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
    JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
    LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_SS_COUNTRY_ID = C.SS_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent' {source_filter}
    group by 1,2 HAVING PV > 0;"""

c_a5s_by_country_tch = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_content_table_name} where pageviews > 0) select first_country, topic_channel,
    count(distinct COALESCE_UID) as user_count,
    sum(pageviews) as pv
FROM t1
where d between '{start_date}' and '{end_date}' group by 1,2;"""

# A6. by region
c_a6_by_region = f"""select 
    c.region as first_region,
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', e.SS_event_id, null)) as pv,
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
    JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
    LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_SS_COUNTRY_ID = C.SS_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent' {source_filter}
    group by 1 HAVING PV > 0;"""

c_a6s_by_region = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_content_table_name} where pageviews > 0)
    select first_region,
    count(distinct COALESCE_UID) as user_count,
    sum(pageviews) as pv
FROM t1
where d between '{start_date}' and '{end_date}' group by 1;"""

# A7. by subschannel
c_a7_by_subchannel = f"""select 
    topic_subchannel,
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', e.SS_event_id, null)) as pv,
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent' {source_filter}
    group by 1 HAVING PV > 0;"""

c_a7s_by_subchannel = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_content_table_name} where pageviews > 0)
    select topic_subchannel,
    count(distinct COALESCE_UID) as user_count,
    sum(pageviews) as pv
FROM t1
where d between '{start_date}' and '{end_date}' group by 1;"""


# A8. by referrer 
c_a8_by_referrer = f"""select 
    COALESCE(TRIM(REGEXP_SUBSTR(FIRST_UTM_REFERRER_NULL_ALLOWED, '^[^-]+')), 'Other') AS FIRST_UTM_REFERRER_SIMPLIFIED, 
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', e.SS_event_id, null)) as pv,
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
    JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent' {source_filter}
    group by 1 HAVING PV > 0;"""

c_a8s_by_referrer = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_content_table_name} where pageviews > 0) select first_utm_referrer_simplified,
    count(distinct COALESCE_UID) as user_count,
    sum(pageviews) as pv
FROM t1
where d between '{start_date}' and '{end_date}' group by 1;"""

# A9. By edition 
c_a9_by_edition = f"""select 
    ED.EDITION AS FIRST_EDITION,
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', e.SS_event_id, null)) as pv,
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
    JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
    LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.SS_EDITION_ID = ED.SS_EDITION_ID
where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent' {source_filter}
    group by 1 HAVING PV > 0;"""

c_a9s_by_edition = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_content_table_name} where pageviews > 0) select FIRST_EDITION,
    count(distinct COALESCE_UID) as user_count,
    sum(pageviews) as pv
FROM t1 
where d between '{start_date}' and '{end_date}' group by 1;"""

# A10. By continent
c_a10_by_continent = f"""select 
    C.REGION_CONTINENT AS FIRST_CONTINENT,
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', e.SS_event_id, null)) as pv,
FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
    JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
    LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_SS_COUNTRY_ID = C.SS_COUNTRY_ID
where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent' {source_filter}
    group by 1 HAVING PV > 0;"""

c_a10s_by_continent = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_content_table_name} where pageviews > 0) select FIRST_CONTINENT,
    count(distinct COALESCE_UID) as user_count,
    sum(pageviews) as pv
FROM t1 
where d between '{start_date}' and '{end_date}' group by 1;"""

# B1. Users and PV of top 10 canonical URL for SET A 
c_b1_top_canonical_url = f"""select CANONICAL_URL,
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', e.SS_event_id, null)) as pv
    FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
    where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent' {source_filter}
        group by 1 ORDER BY PV DESC LIMIT 10;"""
    
c_b1s_top_canonical_url = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_content_table_name} where pageviews > 0) select CANONICAL_URL,
    count(distinct COALESCE_UID) as user_count,
    SUM(PAGEVIEWS) AS PV
FROM T1
where d between '{start_date}' and '{end_date}' group by 1 ORDER BY PV DESC LIMIT 10;"""

# B2. Users and PV of top 10 topic channel for Jan 1-5 & desktop
c_b2_top_topic_channel_by_source = f"""select source as first_source, topic_channel,
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', e.SS_event_id, null)) as pv
    FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
    where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent' {source_filter}
        and source = 'desktop'
        group by 1,2 ORDER BY PV DESC LIMIT 10;"""
    
c_b2s_top_topic_channel_by_source = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_content_table_name} where pageviews > 0) select first_source, topic_channel,
    count(distinct COALESCE_UID) as user_count,
    SUM(PAGEVIEWS) AS PV
FROM T1
where d between '{start_date}' and '{end_date}' and first_source = 'desktop' group by 1,2 ORDER BY PV DESC LIMIT 10;"""

# B3. Users and PV of top 10 landing page for Jan 1-5
c_b3_top_landing_page_user_type = f"""select 
    CASE WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
        WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
        WHEN LOWER(APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
        ELSE NULL END AS user_type,
    content_type,
    CANONICAL_URL, 
    count(distinct iff(event_type = 'page', IFF(LOWER(APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register'), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID), null)) as user_count,
    count(distinct iff(event_type = 'page', SS_event_id, null)) as pv,
    FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
    where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent'
        and user_type = 'subscriber'
        and content_type = 'landing page' {source_filter}
        group by 1,2,3 ORDER BY PV DESC LIMIT 10;"""
    
c_b3s_top_landing_page_user_type = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_content_table_name} where pageviews > 0) select user_type, content_type, CANONICAL_URL,
    COUNT(DISTINCT UID_BY_USERTYPE) AS USER_COUNT,
    SUM(PAGEVIEWS) AS PV
    FROM T1
    where d between '{start_date}' and '{end_date}' 
        and user_type = 'subscriber' and content_type = 'landing page' group by 1,2,3 ORDER BY PV DESC LIMIT 10;"""

# B4. Users and PV of top 10 top article for Jan 1-5
c_b4_top_article = f"""select 
    content_type,
    CANONICAL_URL, 
    count(distinct iff(event_type = 'page', COALESCE(E.USER_DIM_ID, e.SS_anonymous_user_id), null)) as user_count,
    count(distinct iff(event_type = 'page', SS_event_id, null)) as pv,
    FROM mydataspace.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM e
    where e.d between '{start_date}' and '{end_date}' AND SEGMENT_USER_AGENT_TYPE = 'other user agent'
        and content_type = 'article' {source_filter}
        group by 1,2 ORDER BY PV DESC LIMIT 10;"""
    
c_b4s_top_article = f"""with t1 as (select * from MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_content_table_name} where pageviews > 0) select content_type, CANONICAL_URL,
    count(distinct COALESCE_UID) as user_count,
    SUM(PAGEVIEWS) AS PV
    FROM T1
    where d between '{start_date}' and '{end_date}' 
        and content_type = 'article' group by 1,2 ORDER BY PV DESC LIMIT 10;"""

## Daily KPI Review

### SINGLE GROUP BY

In [339]:
cur.execute(q1_overall)
rows = cur.fetchall()
cols = [col[0] for col in cur.description]
df1 = pd.DataFrame(rows, columns=cols)

cur.execute(q1s_overall)
rows = cur.fetchall()
cols = [col[0] for col in cur.description]
df1s = pd.DataFrame(rows, columns=cols)
#df1s = cur.fetch_pandas_all()

In [340]:
df1 = derived_metrics(df1)
df1s = derived_metrics(df1s)

In [341]:
if (df1 == df1s).all().all():
    result = "Daily KPI - A1: Pass"
else:
    result = "Fail"
print(result)
df1 == df1s

Fail


,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH,PV/U,PV/S,sessions/U,bottom_reach_perc,recirculation_perc,video_completion_perc
0,False,True,True,True,True,True,True,False,True,False,True,True,True


In [417]:
df1 # aligned

,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH,PV/U,PV/S,sessions/U,bottom_reach_perc,recirculation_perc,video_completion_perc
0,11259960,21261887,21345911,9674313,803579,918668,4728240,1.895736,1.003952,1.888274,0.221506,0.043037,0.083063


In [418]:
df1s

,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH,PV/U,PV/S,sessions/U,bottom_reach_perc,recirculation_perc,video_completion_perc
0,11259965,21261887,21345911,9674313,803579,918668,4728240,1.895735,1.003952,1.888273,0.221506,0.043037,0.083063


In [342]:
df2 = run_query(q2_daily)
df2s = run_query(q2s_daily)

In [343]:
df3 = run_query(q3_by_source)
df3s = run_query(q3s_by_source)

In [344]:
df4 = run_query(q4_by_country)
df4s = run_query(q4s_by_country)

In [345]:
df5 = run_query(q5_by_user_type)
df5s = run_query(q5s_by_user_type)

In [346]:
df6 = run_query(q6_by_edition)
df6s = run_query(q6s_by_edition)

In [347]:
df7 = run_query(q7_by_referrer)
df7s = run_query(q7s_by_referrer)

In [348]:
df8 = run_query(q8_by_region)
df8s = run_query(q8s_by_region)

In [349]:
df9 = run_query(q9_by_continent)
df9s = run_query(q9s_by_continent)

In [350]:
compare_dataframes(df2, df2s, ['D'], False)

The number of rows in df1 and df2 are 6 and 6. They match.
The discrepancy in metric USER_COUNT ranges between 0.0000% and 0.0001%. The absolute discrepancy is below 0.1%. This is acceptable.

Detailed discrepancies:
             D      metric  df1_value  df2_value  difference  %_DIFFERENCE
0  2026-03-16  USER_COUNT    2494480    2494481          -1      0.000040
1  2026-03-17  USER_COUNT    2657776    2657778          -2      0.000075
2  2026-03-18  USER_COUNT    2539022    2539023          -1      0.000039
3  2026-03-20  USER_COUNT    2175884    2175886          -2      0.000092
The discrepancy in metric USER_COUNT ranges between 0.0000% and 0.0001%. The absolute discrepancy is below 0.1%. ✅ This is acceptable. [PASS]


In [351]:
df2

,D,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,2026-03-19,2549437,3834287,3869750,1804070,126079,153483,810263
1,2026-03-17,2657776,3976821,4037786,1809931,200913,193275,955835
2,2026-03-16,2494480,3754006,3778954,1764771,147054,178704,816049
3,2026-03-18,2539022,3809721,3848108,1693583,135416,158766,896792
4,2026-03-15,1812906,2655001,2574331,1217496,86197,109317,559564
5,2026-03-20,2175884,3232051,3236982,1384462,107920,125123,689737


In [352]:
compare_dataframes(df3, df3s, ['SOURCE'], False)

The number of rows in df1 and df2 are 2 and 2. They match.
The discrepancy in metric USER_COUNT ranges between 0.0000% and 0.0001%. The absolute discrepancy is below 0.1%. This is acceptable.

Detailed discrepancies:
       SOURCE      metric  df1_value  df2_value  difference  %_DIFFERENCE
0    desktop  USER_COUNT    2726598    2726602          -4      0.000147
1  mobileweb  USER_COUNT    8543409    8543410          -1      0.000012
The discrepancy in metric USER_COUNT ranges between 0.0000% and 0.0001%. The absolute discrepancy is below 0.1%. ✅ This is acceptable. [PASS]


In [353]:
df3

,SOURCE,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,desktop,2726598,6084376,7214093,3878886,506774,311109,1272539
1,mobileweb,8543409,15177511,14131818,5795427,296805,607559,3455701


In [354]:
df3s

,SOURCE,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,mobileweb,8543410,15177511,14131818,5795427,296805,607559,3455701
1,desktop,2726602,6084376,7214093,3878886,506774,311109,1272539


In [355]:
compare_dataframes(df5, df5s, ['USER_TYPE'], False)

The number of rows in df1 and df2 are 3 and 3. They match.
The discrepancy in metric SESSIONS ranges between 0.0007% and 0.0438%. The absolute discrepancy is below 0.1%. This is acceptable.

Detailed discrepancies:
     USER_TYPE    metric  df1_value  df2_value  difference  %_DIFFERENCE
0  registered  SESSIONS     583957     583953           4      0.000685
1  subscriber  SESSIONS     331389     331244         145      0.043755
The discrepancy in metric SESSIONS ranges between 0.0007% and 0.0438%. The absolute discrepancy is below 0.1%. ✅ This is acceptable. [PASS]


In [356]:
df5

,USER_TYPE,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,registered,130808,583957,963623,465713,39885,19684,118557
1,subscriber,40111,331389,702575,47635,20677,20148,111568
2,anonymous,11108889,20417391,19679713,9160965,743017,878836,4498115


In [357]:
df5s

,USER_TYPE,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,registered,130808,583953,963623,465713,39885,19684,118557
1,anonymous,11108889,20417391,19679713,9160965,743017,878836,4498115
2,subscriber,40111,331244,702575,47635,20677,20148,111568


In [358]:
compare_dataframes(df6, df6s, ['FIRST_EDITION'], False)

The number of rows in df1 and df2 are 4 and 4. They match.
The discrepancy in metric USER_COUNT ranges between 0.0001% and 0.0001%. The absolute discrepancy is below 0.1%. This is acceptable.

Detailed discrepancies:
   FIRST_EDITION      metric  df1_value  df2_value  difference  %_DIFFERENCE
0            us  USER_COUNT    9506160    9506165          -5      0.000053
The discrepancy in metric USER_COUNT ranges between 0.0001% and 0.0001%. The absolute discrepancy is below 0.1%. ✅ This is acceptable. [PASS]


In [359]:
compare_dataframes(df7, df7s, ['FIRST_UTM_REFERRER_SIMPLIFIED'], False)

The number of rows in df1 and df2 are 9 and 9. They match.
The discrepancy in metric USER_COUNT ranges between 0.0000% and 0.0002%. The absolute discrepancy is below 0.1%. This is acceptable.

Detailed discrepancies:
   FIRST_UTM_REFERRER_SIMPLIFIED      metric  df1_value  df2_value  difference  \
0                        Direct  USER_COUNT    1865770    1865774          -4   
1                       Organic  USER_COUNT    6098384    6098385          -1   

   %_DIFFERENCE  
0      0.000214  
1      0.000016  
The discrepancy in metric USER_COUNT ranges between 0.0000% and 0.0002%. The absolute discrepancy is below 0.1%. ✅ This is acceptable. [PASS]


In [360]:
df7

,FIRST_UTM_REFERRER_SIMPLIFIED,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,Email,19862,29569,25708,18604,1399,155,5736
1,Organic,6098384,9957213,9884021,5010751,374350,517290,2689834
2,Paid,57999,120862,158627,43476,6128,1910,10607
3,Newsletter,49566,88534,107512,59025,7306,2078,26308
4,Other,2691718,4878006,3917024,1715314,111721,97819,956308
5,Social,736314,1164691,970393,521540,30394,23553,293693
6,Eloqua,318,534,799,77,26,8,41
7,Direct,1865770,4591206,5836065,2014971,250563,273052,654337
8,AI Agents,247769,431272,445762,290555,21692,2803,91376


In [361]:
df7s

,FIRST_UTM_REFERRER_SIMPLIFIED,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,Other,2691718,4878006,3917024,1715314,111721,97819,956308
1,AI Agents,247769,431272,445762,290555,21692,2803,91376
2,Organic,6098385,9957213,9884021,5010751,374350,517290,2689834
3,Email,19862,29569,25708,18604,1399,155,5736
4,Paid,57999,120862,158627,43476,6128,1910,10607
5,Direct,1865774,4591206,5836065,2014971,250563,273052,654337
6,Eloqua,318,534,799,77,26,8,41
7,Social,736314,1164691,970393,521540,30394,23553,293693
8,Newsletter,49566,88534,107512,59025,7306,2078,26308


In [362]:
compare_dataframes(df8, df8s, ['FIRST_REGION'], False)

The number of rows in df1 and df2 are 7 and 7. They match.
The discrepancy in metric USER_COUNT ranges between 0.0000% and 0.0001%. The absolute discrepancy is below 0.1%. This is acceptable.

Detailed discrepancies:
   FIRST_REGION      metric  df1_value  df2_value  difference  %_DIFFERENCE
0         APAC  USER_COUNT    1574535    1574536          -1      0.000064
1         EMEA  USER_COUNT    1579886    1579888          -2      0.000127
2           NA  USER_COUNT    6910628    6910630          -2      0.000029
The discrepancy in metric USER_COUNT ranges between 0.0000% and 0.0001%. The absolute discrepancy is below 0.1%. ✅ This is acceptable. [PASS]


In [363]:
compare_dataframes(df4, df4s, ['FIRST_COUNTRY'], False)

The number of rows in df1 and df2 are 238 and 238. They match.
The discrepancy in metric USER_COUNT ranges between 0.0000% and 0.0014%. The absolute discrepancy is below 0.1%. This is acceptable.

Detailed discrepancies:
                                        FIRST_COUNTRY      metric  df1_value  \
0                                              China  USER_COUNT      73831   
1  United Kingdom of Great Britain and Northern I...  USER_COUNT    1153915   
2                           United States of America  USER_COUNT    6421832   

   df2_value  difference  %_DIFFERENCE  
0      73832          -1      0.001354  
1    1153917          -2      0.000173  
2    6421834          -2      0.000031  
The discrepancy in metric USER_COUNT ranges between 0.0000% and 0.0014%. The absolute discrepancy is below 0.1%. ✅ This is acceptable. [PASS]


In [422]:
df8

,FIRST_REGION,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,002,11,20,22,4,1,4,6
1,NA,6910628,12767674,12157896,6458431,524720,520526,2081206
2,150,85,124,121,66,1,1,32
3,EMEA,1579886,3000429,3096413,1664025,108124,62362,552552
4,NaN,1128355,2420894,2610725,432762,82895,178547,927797
5,APAC,1574535,2892983,3282680,1027791,79399,154199,1143372
6,LATAM,78700,179763,198054,91234,8439,3029,23275


In [425]:
df8s

,FIRST_REGION,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,APAC,1574536,2892983,3282680,1027791,79399,154199,1143372
1,LATAM,78700,179763,198054,91234,8439,3029,23275
2,002,11,20,22,4,1,4,6
3,150,85,124,121,66,1,1,32
4,EMEA,1579888,3000429,3096413,1664025,108124,62362,552552
5,NaN,1128355,2420894,2610725,432762,82895,178547,927797
6,NA,6910630,12767674,12157896,6458431,524720,520526,2081206


In [426]:
print(f"FIRST_REGION & FIRST_COUNTRY is NULL for {2610725/21345911} PVs")

FIRST_REGION & FIRST_COUNTRY is NULL for 0.12230562565355023 PVs


In [364]:
df4

,FIRST_COUNTRY,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,New Caledonia,1,1,1,0,0,0,0
1,United Republic of Tanzania,264,377,409,356,39,9,98
2,Sao Tome and Principe,2,3,1,0,0,0,0
3,Montserrat,1,4,3,6,1,0,0
4,Spain,36425,84273,97156,49004,4019,1912,13239
...,...,...,...,...,...,...,...,...
233,Guyana,76,107,114,94,4,5,34
234,Timor-Leste,1,1,1,0,0,0,1
235,France,44109,88300,95746,44070,3196,2423,15314
236,Canary Islands,11,20,22,4,1,4,6


In [424]:
df4[df4['FIRST_COUNTRY'].isna()]

,FIRST_COUNTRY,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
160,NaN,1128355,2420894,2610725,432762,82895,178547,927797


In [366]:
compare_dataframes(df9, df9s, ['FIRST_CONTINENT'], False)

The number of rows in df1 and df2 are 6 and 6. They match.
The discrepancy in metric USER_COUNT ranges between 0.0000% and 0.0001%. The absolute discrepancy is below 0.1%. This is acceptable.

Detailed discrepancies:
   FIRST_CONTINENT      metric  df1_value  df2_value  difference  %_DIFFERENCE
0        Americas  USER_COUNT    6989259    6989261          -2      0.000029
1            Asia  USER_COUNT    1341851    1341852          -1      0.000075
2          Europe  USER_COUNT    1477354    1477356          -2      0.000135
The discrepancy in metric USER_COUNT ranges between 0.0000% and 0.0001%. The absolute discrepancy is below 0.1%. ✅ This is acceptable. [PASS]


In [423]:
df9s

,FIRST_CONTINENT,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,NaN,1128355,2420894,2610725,432762,82895,178547,927797
1,Americas,6989261,12947483,12356008,6549689,533161,523557,2104489
2,Africa,63297,95073,100769,66424,3521,2805,25129
3,Oceania,272259,465710,442364,227317,10649,6905,77617
4,Asia,1341852,2502064,2933843,831697,70671,149999,1082799
5,Europe,1477356,2830663,2902202,1566424,102682,56855,510409


In [367]:
# E2.Sessions by referrer pie chart PASS: 2025-11-03, 2025-11-09
# Calculate total sum of sessions
total_sessions = df7['SESSIONS'].sum()

# Calculate percentage for each value
df7_pie = df7.copy()
df7_pie['SESSIONS_PERCENT'] = (df7_pie['SESSIONS'] / total_sessions) * 100
df7_pie

,FIRST_UTM_REFERRER_SIMPLIFIED,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH,SESSIONS_PERCENT
0,Email,19862,29569,25708,18604,1399,155,5736,0.139070
1,Organic,6098384,9957213,9884021,5010751,374350,517290,2689834,46.831276
2,Paid,57999,120862,158627,43476,6128,1910,10607,0.568444
3,Newsletter,49566,88534,107512,59025,7306,2078,26308,0.416398
4,Other,2691718,4878006,3917024,1715314,111721,97819,956308,22.942489
5,Social,736314,1164691,970393,521540,30394,23553,293693,5.477835
6,Eloqua,318,534,799,77,26,8,41,0.002512
7,Direct,1865770,4591206,5836065,2014971,250563,273052,654337,21.593596
8,AI Agents,247769,431272,445762,290555,21692,2803,91376,2.028381


### MULTIPLE GROUP BY

In [368]:
dfb1 = run_query(b1_by_source_referrer)
dfb1s = run_query(b1s_by_source_referrer)

In [369]:
dfc1 = run_query(c1_by_day_source)
dfc1s = run_query(c1s_by_day_source)

In [370]:
compare_dataframes(dfb1, dfb1s, ['SOURCE', 'FIRST_UTM_REFERRER_SIMPLIFIED'], False)

The number of rows in df1 and df2 are 18 and 18. They match.
The discrepancy in metric USER_COUNT ranges between 0.0001% and 0.0004%. The absolute discrepancy is below 0.1%. This is acceptable.

Detailed discrepancies:
       SOURCE FIRST_UTM_REFERRER_SIMPLIFIED      metric  df1_value  df2_value  \
0    desktop                        Direct  USER_COUNT     846808     846811   
1    desktop                       Organic  USER_COUNT    1387618    1387619   
2  mobileweb                        Direct  USER_COUNT    1023536    1023537   

   difference  %_DIFFERENCE  
0          -3      0.000354  
1          -1      0.000072  
2          -1      0.000098  
The discrepancy in metric USER_COUNT ranges between 0.0001% and 0.0004%. The absolute discrepancy is below 0.1%. ✅ This is acceptable. [PASS]


In [371]:
# C1. by_source_referrer_edition
compare_dataframes(dfc1, dfc1s, ['SOURCE', 'D'], False)

The number of rows in df1 and df2 are 12 and 12. They match.
The discrepancy in metric USER_COUNT ranges between 0.0001% and 0.0003%. The absolute discrepancy is below 0.1%. This is acceptable.

Detailed discrepancies:
       SOURCE           D      metric  df1_value  df2_value  difference  \
0    desktop  2026-03-16  USER_COUNT     694734     694735          -1   
1    desktop  2026-03-17  USER_COUNT     703620     703622          -2   
2    desktop  2026-03-18  USER_COUNT     664717     664718          -1   
3    desktop  2026-03-20  USER_COUNT     529527     529528          -1   
4  mobileweb  2026-03-20  USER_COUNT    1648377    1648378          -1   

   %_DIFFERENCE  
0      0.000144  
1      0.000284  
2      0.000150  
3      0.000189  
4      0.000061  
The discrepancy in metric USER_COUNT ranges between 0.0001% and 0.0003%. The absolute discrepancy is below 0.1%. ✅ This is acceptable. [PASS]


In [372]:
dfc2 = run_query(c2_by_day_usertype)
dfc2s = run_query(c2s_by_day_usertype)

In [373]:
dfc3 = run_query(c3_by_source_usertype)
dfc3s = run_query(c3s_by_source_usertype)

In [374]:
dfc4 = run_query(c4_by_usertype_referrer)
dfc4s = run_query(c4s_by_usertype_referrer)

In [375]:
dfc5 = run_query(c5_by_usertype_country)
dfc5s = run_query(c5s_by_usertype_country)

In [376]:
dfd1 = run_query(d1_by_day_source_referrer_edition)
dfd1s = run_query(d1s_by_day_source_referrer_edition)
dfd2 = run_query(d2_by_day_source_referrer_region)
dfd2s = run_query(d2s_by_day_source_referrer_region)

In [377]:
dfe1 = run_query(e1) # e1: date_set, source, first_region 
dfe1s = run_query(e1s)

In [378]:
# c2_by_day_usertype - PASS
compare_dataframes(dfc2, dfc2s, ['D', 'USER_TYPE'])

The number of rows in df1 and df2 are 18 and 18. They match.
The discrepancy in metric SESSIONS ranges between 0.0009% and 0.0602%. The absolute discrepancy is below 0.1%. This is acceptable.

Detailed discrepancies:
             D   USER_TYPE    metric  df1_value  df2_value  difference  \
0  2026-03-15  subscriber  SESSIONS      33933      33918          15   
1  2026-03-16  registered  SESSIONS     107740     107738           2   
2  2026-03-16  subscriber  SESSIONS      60099      60080          19   
3  2026-03-17  registered  SESSIONS     100188     100187           1   
4  2026-03-17  subscriber  SESSIONS      60492      60472          20   
5  2026-03-18  registered  SESSIONS     106203     106202           1   
6  2026-03-18  subscriber  SESSIONS      60668      60644          24   
7  2026-03-19  subscriber  SESSIONS      61358      61324          34   
8  2026-03-20  subscriber  SESSIONS      54839      54806          33   

   %_DIFFERENCE  
0      0.044205  
1      0.001856

In [379]:
# c3_by_source_usertype
compare_dataframes(dfc3, dfc3s, ['SOURCE', 'USER_TYPE'])

The number of rows in df1 and df2 are 6 and 6. They match.
The discrepancy in metric SESSIONS ranges between 0.0012% and 0.0527%. The absolute discrepancy is below 0.1%. This is acceptable.

Detailed discrepancies:
       SOURCE   USER_TYPE    metric  df1_value  df2_value  difference  \
0    desktop  registered  SESSIONS     320124     320120           4   
1    desktop  subscriber  SESSIONS     237312     237187         125   
2  mobileweb  subscriber  SESSIONS      94077      94057          20   

   %_DIFFERENCE  
0      0.001250  
1      0.052673  
2      0.021259  
The discrepancy in metric SESSIONS ranges between 0.0012% and 0.0527%. The absolute discrepancy is below 0.1%. ✅ This is acceptable. [PASS]


In [380]:
# c4_by_usertype_referrer
compare_dataframes(dfc4, dfc4s, ['FIRST_UTM_REFERRER_SIMPLIFIED', 'USER_TYPE'])

The number of rows in df1 and df2 are 27 and 27. They match.
The discrepancy in metric SESSIONS ranges between 0.0009% and 0.2807%. The absolute discrepancy is above 0.1%. This is not acceptable.

Detailed discrepancies:
   FIRST_UTM_REFERRER_SIMPLIFIED   USER_TYPE    metric  df1_value  df2_value  \
0                     AI Agents  subscriber  SESSIONS       3563       3553   
1                        Direct  registered  SESSIONS     309958     309955   
2                        Direct  subscriber  SESSIONS     221350     221274   
3                    Newsletter  subscriber  SESSIONS      12154      12150   
4                       Organic  subscriber  SESSIONS      48667      48621   
5                         Other  registered  SESSIONS     117255     117254   
6                         Other  subscriber  SESSIONS      36150      36143   
7                        Social  subscriber  SESSIONS       5932       5930   

   difference  %_DIFFERENCE  
0          10      0.280662  
1     

Note - the 0.22% discrepancy in sessions is driven by 10 SESSION_IDs (3/1360). This should be acceptable. 

In [381]:
# c5_by_usertype_country
compare_dataframes(dfc5, dfc5s, ['FIRST_COUNTRY', 'USER_TYPE'])

The number of rows in df1 and df2 are 470 and 470. They match.
The discrepancy in metric SESSIONS ranges between 0.0015% and 0.7092%. The absolute discrepancy is above 0.1%. This is not acceptable.

Detailed discrepancies:
                                         FIRST_COUNTRY   USER_TYPE    metric  \
0                                           Australia  subscriber  SESSIONS   
1                                            Bulgaria  subscriber  SESSIONS   
2                                              Canada  subscriber  SESSIONS   
3                                               China  subscriber  SESSIONS   
4                                             Germany  subscriber  SESSIONS   
5                                             Ireland  subscriber  SESSIONS   
6                     Republic of Korea (South Korea)  subscriber  SESSIONS   
7   United Kingdom of Great Britain and Northern I...  subscriber  SESSIONS   
8                            United States of America  registered

Note - the discrepancy in sessions is driven by <100 SESSION_IDs (16/4000). This should be acceptable. 

In [382]:
compare_dataframes(dfd1, dfd1s, ['D', 'SOURCE', 'FIRST_UTM_REFERRER_SIMPLIFIED', 'FIRST_EDITION'], False)

The number of rows in df1 and df2 are 324 and 324. They match.
The discrepancy in metric USER_COUNT ranges between 0.0004% and 0.0006%. The absolute discrepancy is below 0.1%. This is acceptable.

Detailed discrepancies:
             D     SOURCE FIRST_UTM_REFERRER_SIMPLIFIED FIRST_EDITION  \
0  2026-03-16    desktop                        Direct            us   
1  2026-03-17    desktop                        Direct            us   
2  2026-03-17    desktop                       Organic            us   
3  2026-03-18    desktop                        Direct            us   
4  2026-03-20    desktop                        Direct            us   
5  2026-03-20  mobileweb                        Direct            us   

       metric  df1_value  df2_value  difference  %_DIFFERENCE  
0  USER_COUNT     208504     208505          -1      0.000480  
1  USER_COUNT     211898     211899          -1      0.000472  
2  USER_COUNT     275932     275933          -1      0.000362  
3  USER_COUNT    

In [383]:
# d2_by_day_source_usertype_referrer_region
d2r = compare_dataframes(dfd2, dfd2s, ['D', 'SOURCE', 'FIRST_UTM_REFERRER_SIMPLIFIED', 'FIRST_REGION'], False)

The number of rows in df1 and df2 are 564 and 564. They match.
The discrepancy in metric USER_COUNT ranges between 0.0005% and 0.0056%. The absolute discrepancy is below 0.1%. This is acceptable.

Detailed discrepancies:
             D     SOURCE FIRST_UTM_REFERRER_SIMPLIFIED FIRST_REGION  \
0  2026-03-16    desktop                        Direct         APAC   
1  2026-03-17    desktop                        Direct           NA   
2  2026-03-17    desktop                       Organic           NA   
3  2026-03-18    desktop                        Direct         EMEA   
4  2026-03-20    desktop                        Direct         EMEA   
5  2026-03-20  mobileweb                        Direct         EMEA   

       metric  df1_value  df2_value  difference  %_DIFFERENCE  
0  USER_COUNT      17739      17740          -1      0.005637  
1  USER_COUNT     150356     150357          -1      0.000665  
2  USER_COUNT     217693     217694          -1      0.000459  
3  USER_COUNT      27712

In [384]:
dfe1 = derived_metrics(dfe1)
dfe1s = derived_metrics(dfe1s)

In [385]:
compare_dataframes(dfe1, dfe1s, ['DATE_SET', 'FIRST_REGION'], content_flag=False)

The number of rows in df1 and df2 are 26 and 26. They match.
The discrepancy in metric USER_COUNT ranges between 0.0631% and 0.3328%. The absolute discrepancy is above 0.1%. This is not acceptable.

Detailed discrepancies:
   DATE_SET FIRST_REGION      metric  df1_value  df2_value  difference  \
0    set_a         APAC  USER_COUNT     386505     386809        -304   
1    set_a         EMEA  USER_COUNT     363161     363661        -500   
2    set_a        LATAM  USER_COUNT      23296      23352         -56   
3    set_a           NA  USER_COUNT    1412978    1413870        -892   
4    set_a          NaN  USER_COUNT     207808     207970        -162   
5    set_b         APAC  USER_COUNT     472193     472794        -601   
6    set_b         EMEA  USER_COUNT     552455     553662       -1207   
7    set_b        LATAM  USER_COUNT      44468      44616        -148   
8    set_b           NA  USER_COUNT    3104828    3107355       -2527   
9    set_b          NaN  USER_COUNT     376415

In [386]:
# desktop America
dfe1_filtered = dfe1[(dfe1['SOURCE'] == 'desktop') & (dfe1['FIRST_REGION'] == 'NA')]


In [387]:
dfe1_filtered

,DATE_SET,SOURCE,FIRST_REGION,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH,PV/U,PV/S,sessions/U,bottom_reach_perc,recirculation_perc,video_completion_perc
8,set_a,desktop,NA,337842,572746,627349,300210,40692,27699,125226,1.85693,1.095335,1.695307,0.199611,0.044152,0.135545
13,set_b,desktop,NA,521799,877652,918794,727077,151875,53476,267284,1.76082,1.046877,1.681973,0.290907,0.058202,0.208884


In [146]:
# Filter the data
set_a = dfe1_filtered[dfe1_filtered['DATE_SET'] == 'set_a'].reset_index(drop=True)
set_b = dfe1_filtered[dfe1_filtered['DATE_SET'] == 'set_b'].reset_index(drop=True)

change_user = (set_b['USER_COUNT'].fillna(0) - set_a['USER_COUNT'].fillna(0)) / set_a['USER_COUNT'].replace(0, np.nan)
change_pv = (set_b['PV'].fillna(0) - set_a['PV'].fillna(0)) / set_a['PV'].replace(0, np.nan)
change_session = (set_b['SESSIONS'].fillna(0) - set_a['SESSIONS'].fillna(0)) / set_a['SESSIONS'].replace(0, np.nan)

change_pv_s = (set_b['PV/S'].fillna(0) - set_a['PV/S'].fillna(0)) / set_a['PV/S'].replace(0, np.nan)
change_article_recir_perc = (set_b['recirculation_perc'].fillna(0) - set_a['recirculation_perc'].fillna(0))

print("User:", change_user.values, "PV:", change_pv.values, "Sessions:", change_session.values, "PV/S:", change_pv_s.values, "recirc_per:", change_article_recir_perc.values)

User: [0.54450601] PV: [0.46456598] Sessions: [0.53235815] PV/S: [-0.04424042] recirc_per: [0.01404993]


In [147]:
set_a

,DATE_SET,SOURCE,FIRST_REGION,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH,PV/U,PV/S,sessions/U,bottom_reach_perc,recirculation_perc,video_completion_perc
0,set_a,desktop,NA,337842,572746,627349,300210,40692,27699,125226,1.85693,1.095335,1.695307,0.199611,0.044152,0.135545


In [148]:
set_b #OK

,DATE_SET,SOURCE,FIRST_REGION,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH,PV/U,PV/S,sessions/U,bottom_reach_perc,recirculation_perc,video_completion_perc
0,set_b,desktop,NA,521799,877652,918794,727077,151875,53476,267284,1.76082,1.046877,1.681973,0.290907,0.058202,0.208884


In [107]:
(1486829-1623293) / 1623293

-0.08406615441574626

## Daily Content Review

In [388]:
#c1 = run_query(c_a1_overall)
c1s = run_query(c_a1s_overall)

In [389]:
c1 = run_query(c_a1_overall)

In [419]:
c1s

,USER_COUNT,PV
0,11030905,21345911


In [420]:
if (c1 == c1s).all().all():
    result = "Daily Content - C1: Pass"
else:
    result = "Fail"
print(result)

Daily Content - C1: Pass


In [391]:
c2 = run_query(c_a2_by_day)
c2s = run_query(c_a2s_by_day)

In [392]:
# C-A2. content by day - pass
compare_dataframes(c2, c2s, ['D'], True)

The number of rows in df1 and df2 are 6 and 6. They match.


'PASS'

In [188]:
c2s

,D,USER_COUNT,PV
0,2026-01-04,2034468,2839026
1,2026-01-02,1089700,1588995
2,2026-01-05,2575583,3749600
3,2026-01-03,2651481,3751279
4,2026-01-01,1361024,1812390


In [189]:
c2s

,D,USER_COUNT,PV
0,2026-01-04,2034468,2839026
1,2026-01-02,1089700,1588995
2,2026-01-05,2575583,3749600
3,2026-01-03,2651481,3751279
4,2026-01-01,1361024,1812390


In [393]:
c3 = run_query(c_a3_by_topic_ch)
c3s = run_query(c_a3s_by_topic_ch)

In [394]:
# C-A3. c_a3_by_topic_ch
compare_dataframes(c3, c3s, ['TOPIC_CHANNEL'], True)

The number of rows in df1 and df2 are 133 and 133. They match.


'PASS'

In [193]:
c3s.head()

,TOPIC_CHANNEL,USER_COUNT,PV
0,soccer,36,37
1,firma,115,193
2,world,5138895,6754317
3,اقتصاد,1967,3923
4,commodities,664,1656


In [395]:
c4 = run_query(c_a4_by_usertype)
c4s = run_query(c_a4s_by_usertype)

In [396]:
# C-A4. c_a4_by_usertype 
compare_dataframes(c4, c4s, ['USER_TYPE'], True)

The number of rows in df1 and df2 are 3 and 3. They match.


'PASS'

In [114]:
c4.head() #OK

,USER_TYPE,USER_COUNT,PV
0,registered,57627,285114
1,subscriber,22017,174998
2,anonymous,3628749,5230270


In [397]:
c5 = run_query(c_a5_by_country_tch)
c5s = run_query(c_a5s_by_country_tch)

In [398]:
# C-A5. c_a5_by_country_tch
compare_dataframes(c5, c5s, ['FIRST_COUNTRY', 'TOPIC_CHANNEL'], True)

The number of rows in df1 and df2 are 4719 and 4719. They match.


'PASS'

In [117]:
c5.head() #OK

,FIRST_COUNTRY,TOPIC_CHANNEL,USER_COUNT,PV
0,South Africa,home,452,1059
1,United Kingdom of Great Britain and Northern I...,sports,10862,12818
2,Singapore,business,4221,5230
3,United States of America,default,2298,2487
4,Taiwan,science,61,100


In [399]:
c6 = run_query(c_a6_by_region)
c6s = run_query(c_a6s_by_region)

In [400]:
# C-A6. c_a6_by_region
compare_dataframes(c6, c6s, ['FIRST_REGION'], True)

The number of rows in df1 and df2 are 7 and 7. They match.


'PASS'

In [120]:
c6.head() #OK

,FIRST_REGION,USER_COUNT,PV
0,None,271228,476754
1,NA,2503274,3683167
2,LATAM,25709,46545
3,EMEA,560114,877649
4,APAC,348218,606267


In [401]:
c7 = run_query(c_a7_by_subchannel)
c7s = run_query(c_a7s_by_subchannel)

In [402]:
compare_dataframes(c7, c7s, ['TOPIC_SUBCHANNEL'], True)

The number of rows in df1 and df2 are 609 and 609. They match.


'PASS'

In [201]:
c7.head() #OK

,TOPIC_SUBCHANNEL,USER_COUNT,PV
0,litigation,108510,122330
1,diversity,20,27
2,wind,40,69
3,world - asia pacific,11,12
4,ホーム,60235,166697


In [403]:
c8 = run_query(c_a8_by_referrer)
c8s = run_query(c_a8s_by_referrer)

In [404]:
compare_dataframes(c8, c8s, ['FIRST_UTM_REFERRER_SIMPLIFIED'], True)

The number of rows in df1 and df2 are 9 and 9. They match.


'PASS'

In [333]:
c8.head() #OK

,FIRST_UTM_REFERRER_SIMPLIFIED,USER_COUNT,PV
0,Newsletter,23214,45551
1,AI Agents,128779,210107
2,Other,2007490,2847932
3,Organic,4897568,6698931
4,Social,419846,529910


In [405]:
c9 = run_query(c_a9_by_edition)
c9s = run_query(c_a9_by_edition)

In [406]:
compare_dataframes(c9, c9s, ['FIRST_EDITION'], True)

The number of rows in df1 and df2 are 4 and 4. They match.


'PASS'

In [129]:
c9.head() #OK

,FIRST_EDITION,USER_COUNT,PV
0,in,1,1
1,None,7094,14579
2,us,3388239,5177464
3,jp,312947,498338


In [407]:
c10 = run_query(c_a10_by_continent)
c10s = run_query(c_a10s_by_continent)

In [408]:
compare_dataframes(c10, c10s, ['FIRST_CONTINENT'], True)

The number of rows in df1 and df2 are 6 and 6. They match.


'PASS'

In [409]:
c_b1 = run_query(c_b1_top_canonical_url)
c_b1s = run_query(c_b1s_top_canonical_url)

In [410]:
c_b2 = run_query(c_b2_top_topic_channel_by_source)
c_b2s = run_query(c_b2s_top_topic_channel_by_source)

In [411]:
c_b3 = run_query(c_b3_top_landing_page_user_type)
c_b3s = run_query(c_b3s_top_landing_page_user_type)

In [412]:
c_b4 = run_query(c_b4_top_article)
c_b4s = run_query(c_b4_top_article)

In [413]:
# C-B1. c_b1_top_canonical_url
compare_dataframes(c_b1, c_b1s, ['CANONICAL_URL'], True)

The number of rows in df1 and df2 are 10 and 10. They match.


'PASS'

In [414]:
# C-B2. c_b2_top_topic_channel_by_source
compare_dataframes(c_b2, c_b2s, ['FIRST_SOURCE', 'TOPIC_CHANNEL'], True)

The number of rows in df1 and df2 are 10 and 10. They match.


'PASS'

In [415]:
# C-B3. c_b3_top_landing_page_user_type
compare_dataframes(c_b3, c_b3s, ['CANONICAL_URL', 'CONTENT_TYPE', 'USER_TYPE'], True)

The number of rows in df1 and df2 are 10 and 10. They match.


'PASS'

In [416]:
# C-B4. c_b4_top_article
compare_dataframes(c_b4, c_b4s, ['CANONICAL_URL', 'CONTENT_TYPE'], True)

The number of rows in df1 and df2 are 10 and 10. They match.


'PASS'

In [213]:
c_b1 #OK

,CANONICAL_URL,USER_COUNT,PV
0,/home/,705656,1708839
1,/world/americas/loud-noises-heard-venezuela-ca...,708432,822089
2,/world/us/venezuelas-maduro-custody-trump-says...,630630,691842
3,/world/americas/world-reacts-us-strikes-venezu...,486932,550763
4,/world/china/china-says-it-cannot-accept-count...,416594,451706
5,/world/europe/several-killed-after-explosion-s...,240784,264860
6,/world/americas/venezuela-vice-president-rodri...,188475,204474
7,/world/us/live-trump-says-us-has-captured-vene...,121403,169110
8,/world/,75991,155388
9,/world/india/india-imposes-excise-duty-cigaret...,120811,127886


In [229]:
c_b1s

,CANONICAL_URL,USER_COUNT,PV
0,/home/,705656,1708839
1,/world/americas/loud-noises-heard-venezuela-ca...,708432,822089
2,/world/us/venezuelas-maduro-custody-trump-says...,630630,691842
3,/world/americas/world-reacts-us-strikes-venezu...,486932,550763
4,/world/china/china-says-it-cannot-accept-count...,416594,451706
5,/world/europe/several-killed-after-explosion-s...,240784,264860
6,/world/americas/venezuela-vice-president-rodri...,188475,204474
7,/world/us/live-trump-says-us-has-captured-vene...,121403,169110
8,/world/,75991,155388
9,/world/india/india-imposes-excise-duty-cigaret...,120811,127886


In [214]:
c_b2 #OK

,FIRST_SOURCE,TOPIC_CHANNEL,USER_COUNT,PV
0,desktop,world,829772,1342533
1,desktop,home,317486,738638
2,desktop,business,284407,414705
3,desktop,markets,200799,335026
4,desktop,NaN,95983,165503
5,desktop,ワールド,72311,116837
6,desktop,ホーム,35134,101835
7,desktop,マーケット,58137,100614
8,desktop,technology,54859,82347
9,desktop,legal,64148,79250


In [215]:
c_b3

,USER_TYPE,CONTENT_TYPE,CANONICAL_URL,USER_COUNT,PV
0,subscriber,landing page,/home/,17216,116901
1,subscriber,landing page,/world/,2594,11809
2,subscriber,landing page,/markets/,1159,8924
3,subscriber,landing page,/business/,1378,4426
4,subscriber,landing page,/world/us/,1039,3886
5,subscriber,landing page,/world/china/,596,2418
6,subscriber,landing page,/world/europe/,538,1821
7,subscriber,landing page,/world/americas/,589,1627
8,subscriber,landing page,/business/energy/,471,1509
9,subscriber,landing page,/technology/,402,1492


In [216]:
c_b4

,CONTENT_TYPE,CANONICAL_URL,USER_COUNT,PV
0,article,/world/americas/loud-noises-heard-venezuela-ca...,708432,822089
1,article,/world/us/venezuelas-maduro-custody-trump-says...,630630,691842
2,article,/world/americas/world-reacts-us-strikes-venezu...,486932,550763
3,article,/world/china/china-says-it-cannot-accept-count...,416594,451706
4,article,/world/europe/several-killed-after-explosion-s...,240784,264860
5,article,/world/americas/venezuela-vice-president-rodri...,188475,204474
6,article,/world/us/live-trump-says-us-has-captured-vene...,121403,169110
7,article,/world/india/india-imposes-excise-duty-cigaret...,120811,127886
8,article,/world/us/5WEOERKMKVNKBJ2GEN2DUVQPUY-2026-01-03/,115618,126198
9,article,/world/americas/us-airlines-cancel-flights-aft...,103621,118963


# QA with GA

In [309]:
def compare_ga(df2s, ga, tolerance=0.05):
    """
    Compares metrics between df1 and df2 after a left join on D.
    The GA dataset is sourced from a Google Analytics export and must include Date as the row dimension and Total Users, Views, and Sessions as metrics. 
    The data should be exported to CSV, cleaned by removing headers and total rows, and saved as ga.csv.
    The Snowflake dataset corresponds to df2 and matches the dataset used in the Daily KPI review.
    GA: https://analytics.google.com/analytics/web/#/analysis/a24152976p359830955
    """

    # Clean up 
    ga['Date'] = pd.to_datetime(ga['Date'], format='%Y%m%d').dt.strftime('%Y-%m-%d')
    ga.rename(columns={'Date':'D', 'Total users':'GA_USER_COUNT', 'Views':'GA_PV', 'Sessions':'GA_SESSIONS'}, inplace=True)
    sf = df2s[['D', 'USER_COUNT', 'PV', 'SESSIONS']]
    sf['D'] = sf['D'].astype('string')

    # 1 + 2) Left join
    df = sf.merge(ga, on='D', how='left')

    # 3) Add ratio columns
    df['USER_RATIO'] = df['USER_COUNT'] / df['GA_USER_COUNT']
    df['PV_RATIO'] = df['PV'] / df['GA_PV']
    df['SESSIONS_RATIO'] = df['SESSIONS'] / df['GA_SESSIONS']

    # 4) Summary statistics (averages)
    avg_user = df['USER_RATIO'].mean()
    avg_pv = df['PV_RATIO'].mean()
    avg_sessions = df['SESSIONS_RATIO'].mean()

    # Helper function for pass/fail evaluation
    def evaluate(metric_name, avg_value):
        lower = 1 - tolerance
        upper = 1 + tolerance
        percent_value = avg_value * 100

        if lower <= avg_value <= upper:
            status = "acceptable ✅"
            verdict = "within +-5%"
        else:
            status = "should be discussed further ⚠️"
            verdict = "out of +-5%"

        print(
            f"Average of {metric_name} is {percent_value:.1f}%. "
            f"The discrepancy is {verdict} and this is {status}."
        )

    # 5) Pass / Fail messaging
    evaluate('USER_COUNT/GA_USER_COUNT', avg_user)
    evaluate('PV/GA_PV', avg_pv)
    evaluate('SESSIONS/GA_SESSIONS', avg_sessions)

    df = df.sort_values(by='D', ascending=True)

    return df

In [310]:
ga = pd.read_csv('ga.csv')
compare_ga(df2s, ga, 0.05)

Average of USER_COUNT/GA_USER_COUNT is 97.0%. The discrepancy is within +-5% and this is acceptable ✅.
Average of PV/GA_PV is 98.3%. The discrepancy is within +-5% and this is acceptable ✅.
Average of SESSIONS/GA_SESSIONS is 116.9%. The discrepancy is out of +-5% and this is should be discussed further ⚠️.


,D,USER_COUNT,PV,SESSIONS,GA_USER_COUNT,GA_PV,GA_SESSIONS,USER_RATIO,PV_RATIO,SESSIONS_RATIO
3,2026-01-01,1391078,1812390,1959332,1430403,1880525,1701032,0.972508,0.963768,1.151849
4,2026-01-02,1115950,1588995,1635305,1153136,1627356,1415500,0.967752,0.976427,1.155284
0,2026-01-03,2695516,3751279,4075687,2747085,3795901,3333530,0.981228,0.988245,1.222634
1,2026-01-04,2076732,2839026,3075208,2156863,2862605,2594293,0.962848,0.991763,1.185374
2,2026-01-05,2631113,3749600,3891660,2731172,3778668,3439690,0.963364,0.992307,1.131398


# Note about User Count Discrepancy

On 2026-01-01, the user count from the SILVER_EVENT_DIM doesn't match to the user count from DAILY_KPI_SUMMARY table. This is due to a glitch in raw data.
The user count from SILVER_EVENT_DIM is using COALESCE(USER_DIM_ID, SS_ANONYMOUS_USER_ID). In the DAILY_KPI_SUMMARY table, the calculation is:
COALESCE(UID_BY_USERTYPE, SS_ANONYMOUS_USER_ID) where UID_BY_USERTYPE is USER_DIM_ID for subscribers & registered users and SS_ANONYMOUS_USER_ID for anonymous users. Theoretically, the values should match if this assumption is correct:
USER_DIM_ID is not null for subscribers & registered users 
AND USER_DIM_ID is null for anonymous users. 
In rare cases (1/1,000,000), this assumption is not held and this is causing the discrepancy between the raw table and summary table. Below are examples 

## 1/1,000,000 discrepancy in USER COUNT: USER_DIM_ID is not null for anonymous user 

In [ ]:
dks_ex = f"""SELECT * FROM MYDATASPACE.A208946_REUTERS_ANALYTICS.{daily_kpi_table_name}
WHERE UID_BY_USERTYPE = '553487df49067d6fff7ce8e4fb29b389' OR COALESCE_UID = '553487df49067d6fff7ce8e4fb29b389'
    AND D = '2026-01-01';"""
dks_df = run_query(dks_ex)

In [252]:
dks_df.rename({'COALESCE_UID':'SS_ANONYMOUS_USER_ID'}, axis=1)

,D,SOURCE,SS_ANONYMOUS_USER_ID,UID_BY_USERTYPE,USER_TYPE,FIRST_UTM_REFERRER_SIMPLIFIED,FIRST_EDITION,FIRST_COUNTRY,FIRST_REGION,FIRST_CONTINENT,SESSIONS_BY_USERTYPE,SESSIONS_FOR_USERTYPE_ALL,PAGEVIEWS,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH,ADJ_COALESCE_UID
0,2026-01-01,desktop,553487df49067d6fff7ce8e4fb29b389,553487df49067d6fff7ce8e4fb29b389,anonymous,Direct,us,United States of America,NA,Americas,1,0,0,0,0,0,0,553487df49067d6fff7ce8e4fb29b389
1,2026-01-01,desktop,553487df49067d6fff7ce8e4fb29b389,2ab17de6-12cd-4fbb-9f21-d5ebb8ad9b6e,registered,Direct,us,United States of America,NA,Americas,1,0,4,0,0,0,0,2ab17de6-12cd-4fbb-9f21-d5ebb8ad9b6e
2,2026-01-01,desktop,553487df49067d6fff7ce8e4fb29b389,2ab17de6-12cd-4fbb-9f21-d5ebb8ad9b6e,subscriber,Direct,us,United States of America,NA,Americas,1,1,2,0,0,0,0,2ab17de6-12cd-4fbb-9f21-d5ebb8ad9b6e


The example above shows that the user had three USER_TYPEs on 2026-01-01. UID_BY_USERTYPE is correct as well. 

In [253]:
event_ex = """SELECT DISTINCT D, SOURCE, RECEIVED_AT, SS_ANONYMOUS_USER_ID, USER_DIM_ID, ANONYMOUS_DIM_ID, SS_SESSION_ID, APPLICATION_AUTHENTICATED_STATE, SS_EVENT_ID, EVENT_TYPE FROM MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM
WHERE (USER_DIM_ID = '2ab17de6-12cd-4fbb-9f21-d5ebb8ad9b6e' OR SS_ANONYMOUS_USER_ID = '553487df49067d6fff7ce8e4fb29b389')
    AND D = '2026-01-01' AND SOURCE in ('desktop', 'mobileweb');"""
event_df = run_query(event_ex)

In [256]:
event_df.sort_values(by='RECEIVED_AT', ascending=False).head()

,D,SOURCE,RECEIVED_AT,SS_ANONYMOUS_USER_ID,USER_DIM_ID,ANONYMOUS_DIM_ID,SS_SESSION_ID,APPLICATION_AUTHENTICATED_STATE,SS_EVENT_ID,EVENT_TYPE
7,2026-01-01,desktop,2026-01-01 05:04:44.323000+00:00,553487df49067d6fff7ce8e4fb29b389,2ab17de6-12cd-4fbb-9f21-d5ebb8ad9b6e,7f668c78-0a61-4843-8df4-fa2d2e2bbf2c,7c48eb4bd43d3e87b509e090daee99d0,anonymous,d4f0bdf870c368043548a74bea663ffd,track
10,2026-01-01,desktop,2026-01-01 05:04:44.318000+00:00,553487df49067d6fff7ce8e4fb29b389,2ab17de6-12cd-4fbb-9f21-d5ebb8ad9b6e,7f668c78-0a61-4843-8df4-fa2d2e2bbf2c,7c48eb4bd43d3e87b509e090daee99d0,logged in,9bd940e5caacbb212c4c94a4b4201a07,track
6,2026-01-01,desktop,2026-01-01 05:04:24.809000+00:00,553487df49067d6fff7ce8e4fb29b389,2ab17de6-12cd-4fbb-9f21-d5ebb8ad9b6e,7f668c78-0a61-4843-8df4-fa2d2e2bbf2c,7c48eb4bd43d3e87b509e090daee99d0,logged in,7956ab331717b75489e718604d3c2871,page
28,2026-01-01,desktop,2026-01-01 05:04:24.650000+00:00,553487df49067d6fff7ce8e4fb29b389,2ab17de6-12cd-4fbb-9f21-d5ebb8ad9b6e,7f668c78-0a61-4843-8df4-fa2d2e2bbf2c,7c48eb4bd43d3e87b509e090daee99d0,logged in,1ef03c9965f5910da10c90027ebaf8ae,identify
11,2026-01-01,desktop,2026-01-01 05:04:24.648000+00:00,553487df49067d6fff7ce8e4fb29b389,2ab17de6-12cd-4fbb-9f21-d5ebb8ad9b6e,7f668c78-0a61-4843-8df4-fa2d2e2bbf2c,7c48eb4bd43d3e87b509e090daee99d0,logged in,fb2fbc81f8aaf58bf5494c82156d7f2f,page


In the example above, the user's USER_DIM_ID is not null at RECEIVED_AT = 2026-01-01 05:04:44.323000+00:00, although aa_state is anonymous. 
Due to this, when COALESCE(USER_DIM_ID, SS_ANONYMOUS_USER_ID) is applied, ANONYMOUS ID = '7f668c78-0a61-4843-8df4-fa2d2e2bbf2c' is not taken when the user is in anonymous state. In contrast, the summary table is using COALESCE(UID_BY_USERTYPE, SS_ANONYMOUS_USER_ID) so the user's ANONYMOUS ID = '7f668c78-0a61-4843-8df4-fa2d2e2bbf2c' is included to the user count. Due to this, the summary table's user count is greater than by 1. 

# <1% Discrepancy in SESSIONS by USER_TYPE 
There's a discrepency in SESSIONS for registered & subscribers between the numbers calculated from SILVER_EVENT_DIM and SUMMARY table. The difference is usually below 1%. 

In [427]:
df5

,USER_TYPE,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,registered,130808,583957,963623,465713,39885,19684,118557
1,subscriber,40111,331389,702575,47635,20677,20148,111568
2,anonymous,11108889,20417391,19679713,9160965,743017,878836,4498115


In [428]:
df5s

,USER_TYPE,USER_COUNT,SESSIONS,PV,VIDEO_STORY_START,VIDEO_STORY_FINISHED,ARTICLE_RECIRCULATION,ARTICLE_BOTTOM_REACH
0,registered,130808,583953,963623,465713,39885,19684,118557
1,anonymous,11108889,20417391,19679713,9160965,743017,878836,4498115
2,subscriber,40111,331244,702575,47635,20677,20148,111568


In [430]:
compare_dataframes(df5, df5s, ['USER_TYPE'])

The number of rows in df1 and df2 are 3 and 3. They match.
The discrepancy in metric SESSIONS ranges between 0.0007% and 0.0438%. The absolute discrepancy is below 0.1%. This is acceptable.

Detailed discrepancies:
     USER_TYPE    metric  df1_value  df2_value  difference  %_DIFFERENCE
0  registered  SESSIONS     583957     583953           4      0.000685
1  subscriber  SESSIONS     331389     331244         145      0.043755
The discrepancy in metric SESSIONS ranges between 0.0007% and 0.0438%. The absolute discrepancy is below 0.1%. ✅ This is acceptable. [PASS]


## The below describes when and why edge cases happen with concrete examples. 

In [22]:
def compare_combinations(raw, summary):
    raw["type_session_tuple_raw"] = list(zip(raw["USER_TYPE"], raw["SS_SESSION_ID"]))
    summary["type_session_tuple_dks"] = list(zip(summary["USER_TYPE"], summary["SESSION_ID_BY_USERTYPE"]))
    result = set(raw["type_session_tuple_raw"]) - set(summary["type_session_tuple_dks"])
    return result

In [6]:
query_raw = """select CASE WHEN LOWER(E.APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
                WHEN LOWER(E.APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
                WHEN LOWER(E.APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(E.APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
                ELSE NULL END AS USER_TYPE,
    count(distinct SS_SESSION_ID)
from MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
    where ss_anonymous_user_id in ('a8253d99693b497196abc549a7fe68a1',
'f9df89f12934fa1bf457d387bb34b31e',
'9a2a400d76a8fc870fbd04fe5378fa5f',
'c28bb424b9d0c35ce610e358c65da270',
'faa402a9cc9f81ebbfef6c7bb265a4a3',
'aa5b6fd88edcb3e4e34c085577ce5b15',
'47518c7d426bb3595e3fd9d776fba939',
'26fd32390c625feb002accb863515989',
'1a6638cebe404aa6b9113c427aa43e98') 
    and E.D BETWEEN '2026-01-01' AND '2026-01-01' 
    AND (E.SOURCE IN ('desktop', 'mobileweb') AND SEGMENT_USER_AGENT_TYPE = 'other user agent') group by 1;"""

query_dks = """select USER_TYPE, SUM(SESSIONS_BY_USERTYPE)
from MYDATASPACE.A208946_REUTERS_ANALYTICS.SILVER_DAILY_KPI_SUMMARY
    where ss_anonymous_user_id in ('a8253d99693b497196abc549a7fe68a1',
'f9df89f12934fa1bf457d387bb34b31e',
'9a2a400d76a8fc870fbd04fe5378fa5f',
'c28bb424b9d0c35ce610e358c65da270',
'faa402a9cc9f81ebbfef6c7bb265a4a3',
'aa5b6fd88edcb3e4e34c085577ce5b15',
'47518c7d426bb3595e3fd9d776fba939',
'26fd32390c625feb002accb863515989',
'1a6638cebe404aa6b9113c427aa43e98') 
    and D BETWEEN '2026-01-01' AND '2026-01-01' group by 1;"""

In [6]:
session_raw_df = run_query(query_raw)
session_dks_df = run_query(query_dks)

In [ ]:
session_raw_df # the sessions from the silver_event_dim

,USER_TYPE,COUNT(DISTINCT SS_SESSION_ID)
0,registered,17
1,subscriber,6
2,anonymous,4


In [ ]:
session_dks_df # the sessions from daily kpi summary table 

,USER_TYPE,SUM(SESSIONS_BY_USERTYPE)
0,registered,17
1,anonymous,4
2,subscriber,2


In [16]:
query_detail = """WITH T1 AS (
        SELECT 
            E.D,
            E.SS_ANONYMOUS_USER_ID AS SS_ANONYMOUS_USER_ID,
            E.USER_DIM_ID,
            E.SS_SESSION_ID,
            E.EVENT_TYPE,
            E.EVENT_NAME,
            E.SS_EVENT_ID,
            CASE WHEN LOWER(E.APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
                WHEN LOWER(E.APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
                WHEN LOWER(E.APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(E.APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
                ELSE NULL END AS USER_TYPE,
            IFF(LOWER(E.APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'subscriber', 'register'), E.USER_DIM_ID, E.SS_ANONYMOUS_USER_ID) AS UID_BY_USERTYPE,
            LOWER(E.APPLICATION_AUTHENTICATED_STATE) AS APPLICATION_AUTHENTICATED_STATE,
            S.SOURCE,
            S.FIRST_APPLICATION_AUTHENTICATED_STATE,
            S.FIRST_UTM_REFERRER_NULL_ALLOWED,
            S.FIRST_USER_DIM_ID,
            S.USER_ID_COUNTS,
            S.USER_TYPE_COUNTS,
            CASE WHEN S.FIRST_APPLICATION_AUTHENTICATED_STATE_NULL_ALLOWED = 'subscriber' THEN 'subscriber'
                WHEN S.FIRST_APPLICATION_AUTHENTICATED_STATE_NULL_ALLOWED IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
                WHEN S.FIRST_APPLICATION_AUTHENTICATED_STATE_NULL_ALLOWED = 'anonymous' OR S.FIRST_APPLICATION_AUTHENTICATED_STATE_NULL_ALLOWED IS NULL THEN 'anonymous'
                ELSE NULL END AS FIRST_USER_TYPE,      
            --ED.EDITION AS FIRST_EDITION,
            --C.COUNTRY_NAME AS FIRST_COUNTRY,
            --C.REGION AS FIRST_REGION,
            --C.REGION_CONTINENT AS FIRST_CONTINENT
        FROM MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
        JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_SESSION_DIM S ON E.SS_SESSION_ID = S.SS_SESSION_ID
        --LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_COUNTRY_DIM C ON S.FIRST_SS_COUNTRY_ID = C.SS_COUNTRY_ID
        --LEFT JOIN MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EDITION_DIM ED ON S.SS_EDITION_ID = ED.SS_EDITION_ID
        WHERE E.D = '2026-01-01'
            AND (E.SOURCE IN ('desktop', 'mobileweb') AND SEGMENT_USER_AGENT_TYPE = 'other user agent')
            AND E.ss_anonymous_user_id in ('a8253d99693b497196abc549a7fe68a1',
                'f9df89f12934fa1bf457d387bb34b31e',
                '9a2a400d76a8fc870fbd04fe5378fa5f',
                'c28bb424b9d0c35ce610e358c65da270',
                'faa402a9cc9f81ebbfef6c7bb265a4a3',
                'aa5b6fd88edcb3e4e34c085577ce5b15',
                '47518c7d426bb3595e3fd9d776fba939',
                '26fd32390c625feb002accb863515989',
                '1a6638cebe404aa6b9113c427aa43e98'))
    SELECT 
        DISTINCT D,
        SOURCE,
        SS_ANONYMOUS_USER_ID,
        UID_BY_USERTYPE,
        USER_TYPE,
        --COALESCE(TRIM(REGEXP_SUBSTR(FIRST_UTM_REFERRER_NULL_ALLOWED, ''^[^-]+'')), ''Other'') AS FIRST_UTM_REFERRER_SIMPLIFIED, 
        --FIRST_EDITION,
        --FIRST_COUNTRY,
        --FIRST_REGION,
        --FIRST_CONTINENT,
        SS_SESSION_ID,
        IFF(USER_TYPE = 'anonymous', SS_SESSION_ID, IFF(EQUAL_NULL(FIRST_USER_DIM_ID, USER_DIM_ID), SS_SESSION_ID, NULL)) SESSION_ID_BY_USERTYPE
        /*COUNT(DISTINCT IFF(USER_TYPE IN (''subscriber'', ''registered'') AND E.USER_DIM_ID IS NULL, IFF(USER_ID_COUNTS = 0 AND FIRST_USER_TYPE=USER_TYPE, E.SS_SESSION_ID, NULL), 
        (IFF(USER_TYPE_COUNTS=1, 
            IFF(USER_ID_COUNTS <= 1, E.SS_SESSION_ID,   -- CASE 1: ONE USER TYPE, <= 1 USER DIM ID: ASSIGN SESSION_ID
                IFF(EQUAL_NULL(FIRST_USER_DIM_ID, E.USER_DIM_ID), E.SS_SESSION_ID, NULL)), -- CASE 2: ONE USER TYPE, MULTIPLE NON-NULL UID: ASSIGN SESSION ID IF THE USER_DIM_ID = FIRST NON NULL USER ID
            IFF(USER_ID_COUNTS <= 1, 
                IFF(FIRST_USER_TYPE=USER_TYPE, E.SS_SESSION_ID, NULL), -- CASE 3: MULTIPLE USER TYPES, ZERO OR ONE UID > ASSIGN SESSION_ID IF USER_TYPE IS THE FIRST_USER TYPE  
                -- CASE 4
                IFF(FIRST_USER_TYPE=USER_TYPE AND (EQUAL_NULL(FIRST_USER_DIM_ID, E.USER_DIM_ID) OR (FIRST_USER_TYPE=''anonymous'')), E.SS_SESSION_ID, NULL)
                -- CASE 4: MULTIPLE USER TYPES, MULTIPLE USER ID: ASSIGN SESSION_ID IF USER_TYPE AND USER ID IS THE FIRST USER ID. 
                IFF(FIRST_USER_TYPE=USER_TYPE AND (EQUAL_NULL(FIRST_USER_DIM_ID, E.USER_DIM_ID) OR FIRST_USER_TYPE = ''anonymous''), E.SS_SESSION_ID, NULL)
                ))))) AS SESSIONS_FOR_USERTYPE_ALL,
        --COUNT(DISTINCT IFF(EVENT_TYPE IN (''page''), SS_EVENT_ID, NULL)) AS PAGEVIEWS,
        --COUNT(DISTINCT IFF(EVENT_NAME = ''video_player.story.started'', SS_EVENT_ID, NULL)) AS VIDEO_STORY_START,
        --COUNT(DISTINCT IFF(EVENT_NAME = ''video_player.story.finished'', SS_EVENT_ID, NULL)) AS VIDEO_STORY_FINISHED,
        --COUNT(DISTINCT IFF(EVENT_NAME = ''article.recirculation'', SS_EVENT_ID, NULL)) AS ARTICLE_RECIRCULATION,
        --COUNT(DISTINCT IFF(EVENT_NAME = ''article.content.bottom.visible'', SS_EVENT_ID, NULL)) AS ARTICLE_BOTTOM_REACH*/
    FROM T1 E;"""


raw_session_ids = """select distinct CASE WHEN LOWER(E.APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
                WHEN LOWER(E.APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
                WHEN LOWER(E.APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(E.APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
                ELSE NULL END AS USER_TYPE,
    SS_SESSION_ID AS SS_SESSION_ID
from MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
    where ss_anonymous_user_id in ('a8253d99693b497196abc549a7fe68a1',
'f9df89f12934fa1bf457d387bb34b31e',
'9a2a400d76a8fc870fbd04fe5378fa5f',
'c28bb424b9d0c35ce610e358c65da270',
'faa402a9cc9f81ebbfef6c7bb265a4a3',
'aa5b6fd88edcb3e4e34c085577ce5b15',
'47518c7d426bb3595e3fd9d776fba939',
'26fd32390c625feb002accb863515989',
'1a6638cebe404aa6b9113c427aa43e98') 
    and E.D BETWEEN '2026-01-01' AND '2026-01-01' 
    AND (E.SOURCE IN ('desktop', 'mobileweb') AND SEGMENT_USER_AGENT_TYPE = 'other user agent');"""

In [17]:
raw_session_ids_df = run_query(raw_session_ids)

In [18]:
raw_session_ids_df

,USER_TYPE,SS_SESSION_ID
0,subscriber,42769d1b50ceecea9d311a7ef2cbf32c
1,registered,9a0cd26973fa3257ea3aeaf85c8406e9
2,registered,1ee0e7975c99078bebad8f6042f3cf61
3,registered,d2548530a623d035503cc61cdee40f2b
4,anonymous,c70c7b21e034c9f3b8ebb59c79b628c9
5,registered,d0df64526054a398a8a40b69a7e7d3b1
6,registered,dfe189915cae2951ef6b5f696943ad82
7,registered,57f2421af15278f8c791808b683679d7
8,registered,cea07720ac77905da1991c18c9f7e2cf
9,subscriber,9c0b407290d00efaedf9bb614803dd8c


In [12]:
query_detail_df = run_query(query_detail)

In [13]:
query_detail_df

,D,SOURCE,SS_ANONYMOUS_USER_ID,UID_BY_USERTYPE,USER_TYPE,SS_SESSION_ID,SESSION_ID_BY_USERTYPE
0,2026-01-01,desktop,26fd32390c625feb002accb863515989,6339995c-300a-4616-abe0-dcabc9868039,registered,d2548530a623d035503cc61cdee40f2b,d2548530a623d035503cc61cdee40f2b
1,2026-01-01,desktop,9a2a400d76a8fc870fbd04fe5378fa5f,c5fcdecf-e0e2-47e4-b42b-0a75142cfd48,registered,a5c9b17b863e30da496bdb36c77c406c,a5c9b17b863e30da496bdb36c77c406c
2,2026-01-01,desktop,9a2a400d76a8fc870fbd04fe5378fa5f,c5fcdecf-e0e2-47e4-b42b-0a75142cfd48,registered,9a0cd26973fa3257ea3aeaf85c8406e9,9a0cd26973fa3257ea3aeaf85c8406e9
3,2026-01-01,desktop,1a6638cebe404aa6b9113c427aa43e98,3e5abaf6-bc7c-45c3-b3b2-422cbfa6dc92,registered,20389c2989e052684af7c7523d2d8aaa,NaN
4,2026-01-01,desktop,47518c7d426bb3595e3fd9d776fba939,7c3fcd72-edef-4b53-a0f6-e791170ee6bf,registered,d0df64526054a398a8a40b69a7e7d3b1,d0df64526054a398a8a40b69a7e7d3b1
5,2026-01-01,mobileweb,f9df89f12934fa1bf457d387bb34b31e,c2672e25-9fd3-461f-83f1-88450bf65aa9,registered,dd8b51781e50367b56c1337ecf589d90,dd8b51781e50367b56c1337ecf589d90
6,2026-01-01,desktop,c28bb424b9d0c35ce610e358c65da270,daada5ef-4e93-47b2-9bbe-73e0953d3d09,subscriber,9c0b407290d00efaedf9bb614803dd8c,9c0b407290d00efaedf9bb614803dd8c
7,2026-01-01,desktop,a8253d99693b497196abc549a7fe68a1,a63ed154-17ce-4b2b-b40b-1f8f966cbf68,registered,57f2421af15278f8c791808b683679d7,NaN
8,2026-01-01,desktop,9a2a400d76a8fc870fbd04fe5378fa5f,93694bae-578e-4bad-9751-4671aace7a3a,registered,9a0cd26973fa3257ea3aeaf85c8406e9,NaN
9,2026-01-01,desktop,47518c7d426bb3595e3fd9d776fba939,23d19005-9489-4c9a-9c2c-73814d52cf5d,registered,d0df64526054a398a8a40b69a7e7d3b1,NaN


In [ ]:
# Four session_ids included in the count from the raw dataset but not in the count from the daily kpi. 
result = compare_combinations(raw_session_ids_df, query_detail_df)
result

{('subscriber', '20389c2989e052684af7c7523d2d8aaa'),
 ('subscriber', '57f2421af15278f8c791808b683679d7'),
 ('subscriber', 'b8ac7aaf764a2c81b3b68b8ddff37323'),
 ('subscriber', 'd2548530a623d035503cc61cdee40f2b')}

In [41]:
edge_case = """SELECT DISTINCT SS_SESSION_ID, SS_ANONYMOUS_USER_ID, USER_DIM_ID, APPLICATION_AUTHENTICATED_STATE, CASE WHEN LOWER(E.APPLICATION_AUTHENTICATED_STATE) = 'subscriber' THEN 'subscriber'
                WHEN LOWER(E.APPLICATION_AUTHENTICATED_STATE) IN ('registered', 'logged in', 'cancelled subscriber', 'register') THEN 'registered'
                WHEN LOWER(E.APPLICATION_AUTHENTICATED_STATE) = 'anonymous' OR LOWER(E.APPLICATION_AUTHENTICATED_STATE) IS NULL THEN 'anonymous'
                ELSE NULL END AS USER_TYPE, MIN(received_at), MAX(RECEIVED_AT)
    FROM MYDATASPACE.A208535_REUTERS_SEGMENT_PREPROD.SILVER_EVENT_DIM E
    WHERE SS_SESSION_ID IN ('20389c2989e052684af7c7523d2d8aaa','57f2421af15278f8c791808b683679d7','b8ac7aaf764a2c81b3b68b8ddff37323','d2548530a623d035503cc61cdee40f2b')
        AND D BETWEEN '2026-01-01' AND '2026-01-01' 
        AND (SOURCE IN ('desktop', 'mobileweb') AND SEGMENT_USER_AGENT_TYPE = 'other user agent') GROUP BY 1,2,3,4,5 ORDER BY 1"""

In [42]:
edge_df = run_query(edge_case)

In [43]:
edge_df

,SS_SESSION_ID,SS_ANONYMOUS_USER_ID,USER_DIM_ID,APPLICATION_AUTHENTICATED_STATE,USER_TYPE,MIN(RECEIVED_AT),MAX(RECEIVED_AT)
0,20389c2989e052684af7c7523d2d8aaa,1a6638cebe404aa6b9113c427aa43e98,9335632d-ccda-4c40-b201-6ffb30c1bbf5,logged in,registered,2026-01-01 11:00:21.916000+00:00,2026-01-01 11:02:37.491000+00:00
1,20389c2989e052684af7c7523d2d8aaa,1a6638cebe404aa6b9113c427aa43e98,3e5abaf6-bc7c-45c3-b3b2-422cbfa6dc92,logged in,registered,2026-01-01 11:02:39.596000+00:00,2026-01-01 11:02:39.596000+00:00
2,20389c2989e052684af7c7523d2d8aaa,1a6638cebe404aa6b9113c427aa43e98,3e5abaf6-bc7c-45c3-b3b2-422cbfa6dc92,subscriber,subscriber,2026-01-01 11:03:18.266000+00:00,2026-01-01 11:28:06.367000+00:00
3,57f2421af15278f8c791808b683679d7,a8253d99693b497196abc549a7fe68a1,a63ed154-17ce-4b2b-b40b-1f8f966cbf68,subscriber,subscriber,2026-01-01 02:35:35.194000+00:00,2026-01-01 02:38:17.743000+00:00
4,57f2421af15278f8c791808b683679d7,a8253d99693b497196abc549a7fe68a1,d297e760-ad9c-41b6-84a2-eccc243a21c9,logged in,registered,2026-01-01 02:34:15.758000+00:00,2026-01-01 02:35:22.243000+00:00
5,57f2421af15278f8c791808b683679d7,a8253d99693b497196abc549a7fe68a1,a63ed154-17ce-4b2b-b40b-1f8f966cbf68,logged in,registered,2026-01-01 02:35:25.465000+00:00,2026-01-01 02:37:05.542000+00:00
6,b8ac7aaf764a2c81b3b68b8ddff37323,1a6638cebe404aa6b9113c427aa43e98,3e5abaf6-bc7c-45c3-b3b2-422cbfa6dc92,subscriber,subscriber,2026-01-01 11:03:13.057000+00:00,2026-01-01 11:12:02.334000+00:00
7,b8ac7aaf764a2c81b3b68b8ddff37323,1a6638cebe404aa6b9113c427aa43e98,9335632d-ccda-4c40-b201-6ffb30c1bbf5,logged in,registered,2026-01-01 10:11:37.956000+00:00,2026-01-01 10:41:25.647000+00:00
8,d2548530a623d035503cc61cdee40f2b,26fd32390c625feb002accb863515989,4112e5c3-98b0-469d-bc7f-2fd6f48f3327,subscriber,subscriber,2026-01-01 03:15:17.474000+00:00,2026-01-01 03:15:29.533000+00:00
9,d2548530a623d035503cc61cdee40f2b,26fd32390c625feb002accb863515989,NaN,anonymous,anonymous,2026-01-01 03:08:31.468000+00:00,2026-01-01 03:15:10.548000+00:00


- Some SS_SESSION_IDs are not counted because the user’s USER_DIM_ID changed during the same session.
- A session is only counted when the USER_DIM_ID matches the first USER_DIM_ID seen in that session (that is, when EQUAL_NULL(FIRST_USER_DIM_ID, USER_DIM_ID) is true). This rule is needed because the DKS table counts sessions by UID_BY_USERTYPE and USER_TYPE.
- Without this rule, session counts can be inflated. For example, if a user starts a session as a registered or subscribed user and their USER_DIM_ID changes during the session, the same SS_SESSION_ID can appear multiple times under different USER_DIM_IDs. This would cause one session to be counted more than once.
- By applying the EQUAL_NULL(FIRST_USER_DIM_ID, USER_DIM_ID) condition, each session is counted only once, and inflated session counts are avoided.
Example: 
SS_ANONYMOUS_USER_ID | USER_DIM_ID (UID_BY_USERTYPE) | SS_SESSION_ID | FIRST_USER_DIM_ID | EQUAL_NULL(FIRST_USER_DIM_ID, USER_DIM_ID) | COUNT(DISTINCT SS_SESSION_ID)
A1 | U1 | S1 | U1 | TRUE | 1
A1 | U2 | S1 | U1 | FALSE | 1 
A1 | U3 | S1 | U1 | FALSE | 1
A2 | U4 | S2 | U4 | TRUE | 1 

In the example above, User A1 had only one session, S1, but this can be counted three times in the summary table if EQUAL_NULL(FIRST_USER_DIM_ID, USER_DIM_ID) is not applied. Applying the condition can resolve this issue. 

However, the edge cases happen when a user's USER_DIM_ID changes during a session and USER_TYPE also changes.
SS_SESSION_ID | USER_DIM_ID | USER_TYPE | EQUAL_NULL(FIRST_USER_DIM_ID, USER_DIM_ID) | COUNT IN DKS 
20389c2989e052684af7c7523d2d8aaa | 9335632d-ccda-4c40-b201-6ffb30c1bbf5	| registered | True | 1 
20389c2989e052684af7c7523d2d8aaa | 3e5abaf6-bc7c-45c3-b3b2-422cbfa6dc92	| subscriber | False | 0 

# Notes
- The referrer_sql below is the latest sql for FIRST_UTM_REFERRER_NULL_ALLOWED as of April 10, 2026. Based on the logic, if this value is NULL, I will assign 'Other.'

In [ ]:
referrer_sql = """case
    --Organic
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%organic%' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%google%' and FIRST_REFERRER_NULL_ALLOWED ilike '%news.google.com%') or (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike 'gnews') then 'Organic - Google News'
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%organic%' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%google%' and FIRST_REFERRER_NULL_ALLOWED not ilike '%news.google.com%') or (FIRST_UTM_MEDIUM_NULL_ALLOWED is null and FIRST_UTM_SOURCE_NULL_ALLOWED is null and FIRST_REFERRER_NULL_ALLOWED ilike 'https://www.google.com/%') or FIRST_REFERRER_NULL_ALLOWED ilike '%android-app://com.google.android.googlequicksearchbox/%' then 'Organic - Google'
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%organic%' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%bing%') then 'Organic - Bing'
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%organic%' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%duckduckgo%') then 'Organic - Duckduckgo'
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%organic%' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%yahoo%') then 'Organic - Yahoo'
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%organic%' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%ecosia.org%') then 'Organic - Ecosia.org'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%search.aol.com%' or FIRST_REFERRER_NULL_ALLOWED ilike '%search.aol.com%') then 'Organic - AOL'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%search.naver.com%' or FIRST_REFERRER_NULL_ALLOWED ilike '%search.naver.com%') then 'Organic - Naver'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%qwant.com%' or FIRST_REFERRER_NULL_ALLOWED ilike '%qwant.com%') then 'Organic - Qwant'
    -- Paid
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%paid%' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%gam%') then 'Paid - GAM'
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%paid%' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%fb%|%meta%|%facebook%') then 'Paid - Meta'
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%paid%' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike 'li|%LinkedIn%|%linkedin%') then 'Paid - LinkedIn'
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%paid%' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike 'x|%Twitter%|%twitter%') then 'Paid - Twitter'
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%paid%' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%google%|%Google%') then 'Paid - Google'
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%paid%' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%chatgpt%') then 'Paid - ChatGPT'
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%paid%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%chatgpt%|x|%Twitter%|%twitter%|li|%LinkedIn%|%linkedin%|%gam%|%fb%|%meta%|%facebook%|%google%|%Google%') then 'Paid      - Others'
    --Eloqua
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%eloqua%') then 'Eloqua'
    --newsletter
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%newsletter%' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%sailthru%') then 'Newsletter - '|| initcap(replace(FIRST_UTM_CAMPAIGN_NULL_ALLOWED, '-', ' '))
    --ai agents
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%perplexity%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%techcrunch%|%blog%|%engadget%') or FIRST_REFERRER_NULL_ALLOWED ilike '%perplexity.ai%' then 'AI Agents - Perplexity'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%chatgpt%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%techcrunch%|%blog%|%engadget%|%namepepper%') or FIRST_REFERRER_NULL_ALLOWED ilike '%chatgpt.com%' then 'AI Agents - ChatGPT'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%arc.net%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%techcrunch%|%blog%|%engadget%') or FIRST_REFERRER_NULL_ALLOWED ilike '%arc.net%' then 'AI Agents - Arc.net'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%search.brave.com%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%techcrunch%|%blog%|%engadget%') or FIRST_REFERRER_NULL_ALLOWED ilike '%search.brave.com%' then 'AI Agents - Brave.com'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%you.com%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%techcrunch%|%blog%|%engadget%') or FIRST_REFERRER_NULL_ALLOWED ilike '%you.com%' then 'AI Agents - you.com'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%central.zapier.com%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%techcrunch%|%blog%|%engadget%') or FIRST_REFERRER_NULL_ALLOWED ilike '%central.zapier.com%' then 'AI Agents - Zapier Central'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%agentgpt%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%techcrunch%|%blog%|%engadget%|%namepepper%') or FIRST_REFERRER_NULL_ALLOWED ilike '%agentgpt%' then 'AI Agents - Agentgpt'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%algolia.com%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%techcrunch%|%blog%|%engadget%') or FIRST_REFERRER_NULL_ALLOWED ilike '%algolia.com%' then 'AI Agents - Algolia'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%jasper.ai%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%techcrunch%|%blog%|%engadget%') or FIRST_REFERRER_NULL_ALLOWED ilike '%jasper.ai%' then 'AI Agents - Jasper AI'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%meta.ai%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%techcrunch%|%blog%|%engadget%') or FIRST_REFERRER_NULL_ALLOWED ilike '%meta.ai%' then 'AI Agents - Meta AI'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%copy.ai%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%techcrunch%|%blog%|%engadget%') or FIRST_REFERRER_NULL_ALLOWED ilike '%copy.ai%' then 'AI Agents - Copy AI'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%edgepilot.com%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%techcrunch%|%blog%|%engadget%') or FIRST_REFERRER_NULL_ALLOWED ilike '%edgepilot.com%' then 'AI Agents - Edgepilot'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%openai%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%techcrunch%|%blog%|%engadget%') or FIRST_REFERRER_NULL_ALLOWED ilike '%openai.com%' then 'AI Agents - OpenAI'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%copilot.microsoft.com%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%techcrunch%|%blog%|%engadget%') or FIRST_REFERRER_NULL_ALLOWED ilike '%copilot.microsoft.com%' then 'AI Agents - Microsoft Copilot'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%iask.ai%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%techcrunch%|%blog%|%engadget%') or FIRST_REFERRER_NULL_ALLOWED ilike '%iask.ai%' then 'AI Agents - Iask.ai'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%claude.ai%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%techcrunch%|%blog%|%engadget%') or FIRST_REFERRER_NULL_ALLOWED ilike '%claude.ai%' then 'AI Agents - Anthropic Claude'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%aitastic.app%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%techcrunch%|%blog%|%engadget%') or FIRST_REFERRER_NULL_ALLOWED ilike '%aitastic.app%' then 'AI Agents - Aitastic.app'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%bnngpt.com%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%techcrunch%|%blog%|%engadget%') or FIRST_REFERRER_NULL_ALLOWED ilike '%bnngpt.com%' then 'AI Agents - Bnngpt.com'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%writesonic%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%techcrunch%|%blog%|%engadget%') or FIRST_REFERRER_NULL_ALLOWED ilike '%writesonic.com%' then 'AI Agents - Writesonic'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%chat-gpt%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%techcrunch%|%blog%|%engadget%') or FIRST_REFERRER_NULL_ALLOWED ilike '%chat-gpt.org%' then 'AI Agents - Chatgpt.org'
    --social
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%social%' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%facebook%' or FIRST_REFERRER_NULL_ALLOWED ilike '%facebook.com%') or (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike 'news_tab' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike 'facebook') then 'Social - Facebook'
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%social%' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%twitter%'or FIRST_REFERRER_NULL_ALLOWED ilike 'https://t.co/%') or FIRST_UTM_SOURCE_NULL_ALLOWED='twitter' then 'Social - Twitter'
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%social%' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%inkedin%'or FIRST_REFERRER_NULL_ALLOWED ilike '%linkedin.com%' or FIRST_REFERRER_NULL_ALLOWED ilike '%lnkd.in%') then 'Social - Linkedin'
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%social%' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%tiktok%' or FIRST_REFERRER_NULL_ALLOWED ilike '%tiktok.com%') then 'Social - Tiktok'
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%social%' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%telegram%' or FIRST_REFERRER_NULL_ALLOWED ilike '%t.me%') then 'Social - Telegram'
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%social%' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%bluesky%' or FIRST_REFERRER_NULL_ALLOWED ilike '%go.bsky.app%') then 'Social - Bluesky'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%reddit.com%' or FIRST_REFERRER_NULL_ALLOWED ilike '%reddit.com%') then 'Social - Reddit'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%threads.net%'or FIRST_REFERRER_NULL_ALLOWED ilike '%threads.net%' or FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%threads.com%'or FIRST_REFERRER_NULL_ALLOWED ilike '%threads.com%') then 'Social - Threads'
    --email
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%email%') then 'Email'
    --other
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%applenews%' or FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%apple.news%' or FIRST_REFERRER_NULL_ALLOWED ilike '%apple.news%') then 'Other - Apple News'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%stocks.apple%' or FIRST_REFERRER_NULL_ALLOWED ilike '%stocks.apple%') then 'Other - Stocks Apple'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%https://statics.teams.cdn.office.net/%'or FIRST_REFERRER_NULL_ALLOWED ilike '%https://statics.teams.cdn.office.net/%' or FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%teams.microsoft%'or FIRST_REFERRER_NULL_ALLOWED ilike '%teams.microsoft%') then 'Other - Microsoft Teams'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%nikkei225jp.com%'or FIRST_REFERRER_NULL_ALLOWED ilike '%nikkei225jp.com%') then 'Other - Nikkei225 JP'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%com.slack%'or FIRST_REFERRER_NULL_ALLOWED ilike '%com.slack%' or FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%slack.com%'or FIRST_REFERRER_NULL_ALLOWED ilike '%slack.com%') then 'Other - Slack'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%drudgereport.com%'or FIRST_REFERRER_NULL_ALLOWED ilike '%drudgereport.com%') then 'Other - Drudge Report'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%projects.fivethirtyeight.com%'or FIRST_REFERRER_NULL_ALLOWED ilike '%projects.fivethirtyeight.com%') then 'Other - 538 Interactive'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%citizenfreepress.com%'or FIRST_REFERRER_NULL_ALLOWED ilike '%citizenfreepress.com%') then 'Other - Citizen Free Press'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%realtime-chart.info%'or FIRST_REFERRER_NULL_ALLOWED ilike '%realtime-chart.info%') then 'Other - Realtime-chart.info'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%wikipedia.org%'or FIRST_REFERRER_NULL_ALLOWED ilike '%wikipedia.org%') then 'Other - Wikipedia'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%https://225225.jp/%'or FIRST_REFERRER_NULL_ALLOWED ilike '%https://225225.jp/%') then 'Other - 225225.jp'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%https://ch225.com/%'or FIRST_REFERRER_NULL_ALLOWED ilike '%https://ch225.com/%') then 'Other - ch225.com'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%https://yandex.ru/%'or FIRST_REFERRER_NULL_ALLOWED ilike '%https://yandex.ru/%') then 'Other - Yandex.ru'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%http://www.smartnews.com/%'or FIRST_REFERRER_NULL_ALLOWED ilike '%http://www.smartnews.com/%') then 'Other - Smartnews.com'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%finviz.com%'or FIRST_REFERRER_NULL_ALLOWED ilike '%finviz.com%') then 'Other - Finviz'
    when (FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%ycombinator.com%'or FIRST_REFERRER_NULL_ALLOWED ilike '%ycombinator.com%') then 'Other - Y Combinator'
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%feed%' and FIRST_UTM_SOURCE_NULL_ALLOWED ilike '%feedburner%') then 'Other - Feedburner'
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%feed%' and FIRST_UTM_SOURCE_NULL_ALLOWED not ilike '%feedburner%') then 'Other - Feed not Feedburner'
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED is null and FIRST_UTM_SOURCE_NULL_ALLOWED is null and FIRST_REFERRER_NULL_ALLOWED is null) then 'Other - Unknown' --null values in 'other' category
    --direct
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED ilike '%direct%') or (FIRST_UTM_MEDIUM_NULL_ALLOWED is null and FIRST_UTM_SOURCE_NULL_ALLOWED is null and FIRST_REFERRER_NULL_ALLOWED ilike'%https://www.reuters.com%' or FIRST_REFERRER_NULL_ALLOWED ilike '%reuters.com%') then 'Direct - Direct' --to prioritize last
    -- Organic (referrer-only, no UTMs)  
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED is null and FIRST_UTM_SOURCE_NULL_ALLOWED is null and FIRST_REFERRER_NULL_ALLOWED ilike '%news.google.com%') then 'Organic - Google News'  
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED is null and FIRST_UTM_SOURCE_NULL_ALLOWED is null and FIRST_REFERRER_NULL_ALLOWED ilike '%google.%' and FIRST_REFERRER_NULL_ALLOWED not ilike '%news.google.com%') then 'Organic - Google'  
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED is null and FIRST_UTM_SOURCE_NULL_ALLOWED is null and FIRST_REFERRER_NULL_ALLOWED ilike '%bing.com%') then 'Organic - Bing'  
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED is null and FIRST_UTM_SOURCE_NULL_ALLOWED is null and FIRST_REFERRER_NULL_ALLOWED ilike '%duckduckgo.com%') then 'Organic - Duckduckgo'  
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED is null and FIRST_UTM_SOURCE_NULL_ALLOWED is null and FIRST_REFERRER_NULL_ALLOWED ilike '%yahoo.com%') then 'Organic - Yahoo'  
    when (FIRST_UTM_MEDIUM_NULL_ALLOWED is null and FIRST_UTM_SOURCE_NULL_ALLOWED is null and FIRST_REFERRER_NULL_ALLOWED ilike '%ecosia.org%') then 'Organic - Ecosia.org'  
    else 'Other - Other' end as FIRST_UTM_REFERRER_NULL_ALLOWED,
 """